In [1]:
import torch
import time
import math
import copy as cp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate 
from scipy.interpolate import make_interp_spline
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from scipy import interpolate 
from scipy.interpolate import make_interp_spline
from torchsummary import summary
from torch.autograd import Function
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_curve,auc

D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#cifar10 and cifar100
batch_size = 32 #[transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC)
transform = transforms.Compose([
    transforms.ToTensor(),   #transforms.ToTensor() 将numpy的ndarray或PIL.Image读的图片转换成形状为(C,H, W)的Tensor格式，且/255归一化到[0,1]之间
    transforms.Normalize((0.5071, 0.4866, 0.4409),(0.2673, 0.2564, 0.2762))  
])

train_dataset = datasets.CIFAR100(
    root='../dataset/cifar100',
    train=True,
    download=True,
    transform=transform,
)
 
test_dataset = datasets.CIFAR100(
    root='../dataset/cifar100',
    train=False,
    download=True,
    transform=transform,
)

train_dataset.targets = np.array(train_dataset.targets)
idx = np.logical_and(train_dataset.targets >= 0, train_dataset.targets < 4)
idx = np.where(idx)


train_dataset.data = train_dataset.data[idx[0]]
train_dataset.targets = torch.Tensor(train_dataset.targets[idx[0]])
#print(np.shape(train_dataset.data))

test_dataset.targets = np.array(test_dataset.targets)
idx1 = np.logical_and(test_dataset.targets >= 0, test_dataset.targets < 4)
idx1 = np.where(idx1)


test_dataset.data = test_dataset.data[idx1[0]]
test_dataset.targets = torch.Tensor(test_dataset.targets[idx1[0]])

train_loader = DataLoader(
    dataset=train_dataset,
    shuffle=True,
    batch_size=batch_size
)
 
test_loader = DataLoader(
    dataset=test_dataset,
    shuffle=True,
    batch_size=batch_size
)


Files already downloaded and verified
Files already downloaded and verified


In [20]:
def quantum_normalization(x):
    y = torch.zeros(len(x),len(x[0]))
    if np.shape(x) != (1,len(x[0])):
        x_tot = torch.zeros(len(x))
        for q in range(len(x)):
            for i in range(len(x[0])):
                x_tot[q] = x_tot[q] + x[q][i].item()*x[q][i].item()
            x_tot[q] = torch.sqrt(x_tot[q])
            for j in range(len(x[0])):
                x[q][j] = x[q][j].item()/x_tot[q]
                y[q][j] = x[q][j]

    if np.shape(x) == (1,len(x[0])):
        #print(x)
        #x = x.detach().numpy()
        x_tot1 = torch.zeros(1)
        for i in range(len(x[0])):
            x_tot1 = x_tot1 + x[0][i].item()*x[0][i].item()
        x_tot1 = torch.sqrt(x_tot1)
        #print(x_tot1)
        for j in range(len(x[0])):
            x[0][j] = x[0][j].item()/x_tot1
            y[0][j] = x[0][j]
            
    return y   

def gate_operation_res(x,n,a,k):
    c = torch.zeros(n,2*len(x[0])-3,len(x[0]),len(x[0]))
   # b = torch.Tensor(5).uniform_(-0.01,0.01)
    #m1 = torch.eye(len(x[0]))
    #m2 = torch.zeros(2*len(x[0])-3,len(x[0]),len(x[0]))
    for t in range(n):
        if t == 2:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])+b[i]
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])+b[i]
                iden[i][i+1] = -torch.sin(a[t][i])+b[i]
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])+b[j+3]
                c[t][j+3] = iden
                
        else:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])
                iden[i][i+1] = -torch.sin(a[t][i])
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])
                c[t][j+3] = iden
            
   
    L1 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[0][0],c[0][1]),c[0][2]),c[0][3]),c[0][4])
    L2 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[1][0],c[1][1]),c[1][2]),c[1][3]),c[1][4])
    L3 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[2][0],c[2][1]),c[2][2]),c[2][3]),c[2][4])
    L4 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[3][0],c[3][1]),c[3][2]),c[3][3]),c[3][4])
    L5 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[4][0],c[4][1]),c[4][2]),c[4][3]),c[4][4])
    L6 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[5][0],c[5][1]),c[5][2]),c[5][3]),c[5][4])
    L7 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[6][0],c[6][1]),c[6][2]),c[6][3]),c[6][4])
    L8 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[7][0],c[7][1]),c[7][2]),c[7][3]),c[7][4])
#    L9 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[8][0],c[8][1]),c[8][2]),c[8][3]),c[8][4])
#    L10 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[9][0],c[9][1]),c[9][2]),c[9][3]),c[9][4])

    m1 = torch.matmul(L1,x.T) #列数是batchsize数
    m1 = m1 + k[7]*x.T
    m1.requires_grad_(True)
    
    m2 = torch.matmul(L2,m1) #列数是batchsize数
    m2 = m2+k[0]*m1
    m2.requires_grad_(True)
   
    m3 = torch.matmul(L3,m2) #列数是batchsize数
    m3 = m3+k[1]*m2
    m3.requires_grad_(True)
    
    m4 = torch.matmul(L4,m3) #列数是batchsize数
    m4 = m4+k[2]*m3
    m4.requires_grad_(True)
    
    m5 = torch.matmul(L5,m4) #列数是batchsize数
    m5 = m5+k[3]*m4
    m5.requires_grad_(True)
   
    m6 = torch.matmul(L6,m5) #列数是batchsize数
    m6 = m6+k[4]*m5
    m6.requires_grad_(True)
    
    m7 = torch.matmul(L7,m6) #列数是batchsize数
    m7 = m7+k[5]*m6
    m7.requires_grad_(True)
    
    m8 = torch.matmul(L8,m7) #列数是batchsize数
    m8 = m8+k[6]*m7
    m8.requires_grad_(True)
    
#    m9 = torch.matmul(L9,m8) #列数是batchsize数
#    m9 = m9+m8
#    m9.requires_grad_(True)
    
#    m10 = torch.matmul(L10,m9) #列数是batchsize数
#    m10 = m10+m9
#    m10.requires_grad_(True)
    
    return  m8.T  #行数是batchsize

def measurement(x):
    return x

In [21]:
def gate_operation_res1(x,n,a,k):
    c = torch.zeros(n,2*len(x[0])-3,len(x[0]),len(x[0]))
   # b = torch.Tensor(5).uniform_(-0.01,0.01)
    #m1 = torch.eye(len(x[0]))
    #m2 = torch.zeros(2*len(x[0])-3,len(x[0]),len(x[0]))
    for t in range(n):
        if t == 2:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])+b[i]
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])+b[i]
                iden[i][i+1] = -torch.sin(a[t][i])+b[i]
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])+b[j+3]
                c[t][j+3] = iden
                
        else:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])
                iden[i][i+1] = -torch.sin(a[t][i])
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])
                c[t][j+3] = iden
            
   
    L1 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[0][0],c[0][1]),c[0][2]),c[0][3]),c[0][4])
    L2 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[1][0],c[1][1]),c[1][2]),c[1][3]),c[1][4])
    L3 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[2][0],c[2][1]),c[2][2]),c[2][3]),c[2][4])
    L4 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[3][0],c[3][1]),c[3][2]),c[3][3]),c[3][4])
    L5 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[4][0],c[4][1]),c[4][2]),c[4][3]),c[4][4])
    L6 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[5][0],c[5][1]),c[5][2]),c[5][3]),c[5][4])
    L7 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[6][0],c[6][1]),c[6][2]),c[6][3]),c[6][4])
    L8 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[7][0],c[7][1]),c[7][2]),c[7][3]),c[7][4])
#    L9 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[8][0],c[8][1]),c[8][2]),c[8][3]),c[8][4])
#    L10 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[9][0],c[9][1]),c[9][2]),c[9][3]),c[9][4])

    m1 = torch.matmul(L1,x.T) #列数是batchsize数
    m1.requires_grad_(True)
    
    m2 = torch.matmul(L2,m1) #列数是batchsize数
    #m2 = m2+k[0]*m1
    m2.requires_grad_(True)
   
    m3 = torch.matmul(L3,m2) #列数是batchsize数
    #m3 = m3+k[1]*m2
    m3.requires_grad_(True)
    
    m4 = torch.matmul(L4,m3) #列数是batchsize数
    #m4 = m4+k[2]*m3
    m4.requires_grad_(True)
    
    m5 = torch.matmul(L5,m4) #列数是batchsize数
    #m5 = m5+k[3]*m4
    m5.requires_grad_(True)
   
    m6 = torch.matmul(L6,m5) #列数是batchsize数
    #m6 = m6+k[4]*m5
    m6.requires_grad_(True)
    
    m7 = torch.matmul(L7,m6) #列数是batchsize数
    #m7 = m7+k[5]*m6
    m7.requires_grad_(True)
    
    m8 = torch.matmul(L8,m7) #列数是batchsize数
    m8 = m8+k[6]*x.T
    m8.requires_grad_(True)
    
#    m9 = torch.matmul(L9,m8) #列数是batchsize数
#    m9 = m9+m8
#    m9.requires_grad_(True)
    
#    m10 = torch.matmul(L10,m9) #列数是batchsize数
#    m10 = m10+m9
#    m10.requires_grad_(True)
    
    return  m8.T  #行数是batchsize

In [22]:
def gate_operation_res2(x,n,a,k):
    c = torch.zeros(n,2*len(x[0])-3,len(x[0]),len(x[0]))
   # b = torch.Tensor(5).uniform_(-0.01,0.01)
    #m1 = torch.eye(len(x[0]))
    #m2 = torch.zeros(2*len(x[0])-3,len(x[0]),len(x[0]))
    for t in range(n):
        if t == 2:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])+b[i]
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])+b[i]
                iden[i][i+1] = -torch.sin(a[t][i])+b[i]
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])+b[j+3]
                c[t][j+3] = iden
                
        else:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])
                iden[i][i+1] = -torch.sin(a[t][i])
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])
                c[t][j+3] = iden
            
   
    L1 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[0][0],c[0][1]),c[0][2]),c[0][3]),c[0][4])
    L2 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[1][0],c[1][1]),c[1][2]),c[1][3]),c[1][4])
    L3 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[2][0],c[2][1]),c[2][2]),c[2][3]),c[2][4])
    L4 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[3][0],c[3][1]),c[3][2]),c[3][3]),c[3][4])
    L5 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[4][0],c[4][1]),c[4][2]),c[4][3]),c[4][4])
    L6 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[5][0],c[5][1]),c[5][2]),c[5][3]),c[5][4])
    L7 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[6][0],c[6][1]),c[6][2]),c[6][3]),c[6][4])
    L8 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[7][0],c[7][1]),c[7][2]),c[7][3]),c[7][4])
#    L9 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[8][0],c[8][1]),c[8][2]),c[8][3]),c[8][4])
#    L10 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[9][0],c[9][1]),c[9][2]),c[9][3]),c[9][4])

    m1 = torch.matmul(L1,x.T) #列数是batchsize数
    m1.requires_grad_(True)
    
    m2 = torch.matmul(L2,m1) #列数是batchsize数
    m2 = m2+k[0]*x.T
    m2.requires_grad_(True)
   
    m3 = torch.matmul(L3,m2) #列数是batchsize数
    #m3 = m3+k[1]*m2
    m3.requires_grad_(True)
    
    m4 = torch.matmul(L4,m3) #列数是batchsize数
    m4 = m4+k[2]*m2
    m4.requires_grad_(True)
    
    m5 = torch.matmul(L5,m4) #列数是batchsize数
    #m5 = m5+k[3]*m4
    m5.requires_grad_(True)
   
    m6 = torch.matmul(L6,m5) #列数是batchsize数
    m6 = m6+k[4]*m4
    m6.requires_grad_(True)
    
    m7 = torch.matmul(L7,m6) #列数是batchsize数
    #m7 = m7+k[5]*m6
    m7.requires_grad_(True)
    
    m8 = torch.matmul(L8,m7) #列数是batchsize数
    m8 = m8+k[6]*m6
    m8.requires_grad_(True)
    
#    m9 = torch.matmul(L9,m8) #列数是batchsize数
#    m9 = m9+m8
#    m9.requires_grad_(True)
    
#    m10 = torch.matmul(L10,m9) #列数是batchsize数
#    m10 = m10+m9
#    m10.requires_grad_(True)
    
    return  m8.T  #行数是batchsize

In [23]:
def gate_operation_dense(x,n,a,k):
    c = torch.zeros(n,2*len(x[0])-3,len(x[0]),len(x[0]))
    #m1 = torch.eye(len(x[0]))
    #m2 = torch.zeros(2*len(x[0])-3,len(x[0]),len(x[0]))
    for t in range(n):
        if t == 2:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])+b[i]
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])+b[i]
                iden[i][i+1] = -torch.sin(a[t][i])+b[i]
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])+b[j+3]
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])+b[j+3]
                c[t][j+3] = iden
                
        else:
            for i in range(len(x[0])-1):
                iden = torch.eye(len(x[0]))
                iden[i+1][i] = torch.sin(a[t][i])
                iden[i][i] = iden[i+1][i+1] = torch.cos(a[t][i])
                iden[i][i+1] = -torch.sin(a[t][i])
                c[t][i] = iden

            for j in range(len(x[0])-2):
                iden = torch.eye(len(x[0]))
                iden[len(x[0])-2-j][len(x[0])-3-j] = torch.sin(a[t][(j+3)])
                iden[len(x[0])-2-j][len(x[0])-2-j] = iden[len(x[0])-3-j][len(x[0])-3-j] = torch.cos(a[t][(j+3)])
                iden[len(x[0])-3-j][len(x[0])-2-j] = -torch.sin(a[t][(j+3)])
                c[t][j+3] = iden
            
   
    L1 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[0][0],c[0][1]),c[0][2]),c[0][3]),c[0][4])
    L2 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[1][0],c[1][1]),c[1][2]),c[1][3]),c[1][4])
    L3 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[2][0],c[2][1]),c[2][2]),c[2][3]),c[2][4])
    L4 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[3][0],c[3][1]),c[3][2]),c[3][3]),c[3][4])
    L5 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[4][0],c[4][1]),c[4][2]),c[4][3]),c[4][4])
    L6 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[5][0],c[5][1]),c[5][2]),c[5][3]),c[5][4])
    L7 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[6][0],c[6][1]),c[6][2]),c[6][3]),c[6][4])
    L8 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[7][0],c[7][1]),c[7][2]),c[7][3]),c[7][4])
#    L9 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[8][0],c[8][1]),c[8][2]),c[8][3]),c[8][4])
#    L10 = torch.matmul(torch.matmul(torch.matmul(torch.matmul(c[9][0],c[9][1]),c[9][2]),c[9][3]),c[9][4])

    m1 = torch.matmul(L1,x.T) #列数是batchsize数
    m1 = m1 + k[7]*x.T
    m1.requires_grad_(True)
    
    m2 = torch.matmul(L2,m1) #列数是batchsize数
    m2 = m2+k[0]*m1
    m2.requires_grad_(True)
   
    m3 = torch.matmul(L3,m2) #列数是batchsize数
    m3 = m3+k[2]*m2+k[1]*m1
    m3.requires_grad_(True)
    
    m4 = torch.matmul(L4,m3) #列数是batchsize数
    m4 = m4+k[5]*m3+k[4]*m2+k[3]*m1
    m4.requires_grad_(True)
    
    m5 = torch.matmul(L5,m4) #列数是batchsize数
    m5 = m5+k[9]*m4+k[8]*m3+k[7]*m2+k[6]*m1
    m5.requires_grad_(True)
   
    m6 = torch.matmul(L6,m5) #列数是batchsize数
    m6 = m6+k[14]*m5+k[13]*m4+k[12]*m3+k[11]*m2+k[10]*m1
    m6.requires_grad_(True)
    
    m7 = torch.matmul(L7,m6) #列数是batchsize数
    m7 = m7+k[20]*m6+k[19]*m5+k[18]*m4+k[17]*m3+k[16]*m2+k[15]*m1
    m7.requires_grad_(True)
    
    m8 = torch.matmul(L8,m7) #列数是batchsize数
    m8 = m8+k[27]*m7+k[26]*m6+k[25]*m5+k[24]*m4+k[23]*m3+k[22]*m2+k[21]*m1
    m8.requires_grad_(True)
    
#    m9 = torch.matmul(L9,m8) #列数是batchsize数
#    m9 = m9+m8
#    m9.requires_grad_(True)
    
#    m10 = torch.matmul(L10,m9) #列数是batchsize数
#    m10 = m10+m9
#    m10.requires_grad_(True)
    
    return  m8.T  #行数是batchsize

In [24]:
class rotate_psi(Function):
        
    @staticmethod
    def forward(ctx,input_,A,a,psi):
        ctx.A = A
        ctx.a = a
        ctx.psi = psi
        y0 = np.sin(a*input_)
        y = A*y0
        x_new = input_*np.cos(psi) - y*np.sin(psi)
        y_new = input_*np.sin(psi) + y*np.cos(psi)
        result = y_new
        ctx.save_for_backward(input_,result)
        return result
    
    @staticmethod    
    def backward(ctx,grad_output,retain_graph=None, create_graph=False): #grad_output = d(loss)/d(result)
        input_,result = ctx.saved_tensors
       # A = 1
       # a = 0.8
       # psi = math.pi/4
        return grad_output*(np.sin(ctx.psi)+ctx.A*ctx.a*np.cos(ctx.a*input_)*np.cos(ctx.psi)),None,None,None

#class rotate_activation(nn.Module):
   
        
   # def forward(self,input_):
    #    return rotate_psi.apply(input_,self.A,self.a,self.psi)
def rotation(input_,A,a,psi):
    return rotate_psi().apply(input_,A,a,psi)

In [25]:
class channel_attention(nn.Module):
    # 初始化, in_channel代表输入特征图的通道数, ratio代表第一个全连接的通道下降倍数
    def __init__(self, in_channel, ratio=4):
        # 继承父类初始化方法
        super(channel_attention, self).__init__()
        
        # 全局最大池化 [b,c,h,w]==>[b,c,1,1]
        self.max_pool = nn.AdaptiveMaxPool2d(output_size=1)
        # 全局平均池化 [b,c,h,w]==>[b,c,1,1]
        self.avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
        
        # 第一个全连接层, 通道数下降4倍
        self.fc1 = nn.Linear(in_features=in_channel, out_features=in_channel//ratio, bias=False)
        # 第二个全连接层, 恢复通道数
        self.fc2 = nn.Linear(in_features=in_channel//ratio, out_features=in_channel, bias=False)
        
        # relu激活函数
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.5)
        # sigmoid激活函数
        self.sigmoid = nn.Sigmoid()
    
    # 前向传播
    def forward(self, inputs):
        # 获取输入特征图的shape
        b, c, h, w = inputs.shape
        
        # 输入图像做全局最大池化 [b,c,h,w]==>[b,c,1,1]
        max_pool = self.max_pool(inputs)
        #print(max_pool.shape)
        # 输入图像的全局平均池化 [b,c,h,w]==>[b,c,1,1]
        avg_pool = self.avg_pool(inputs)
 
        # 调整池化结果的维度 [b,c,1,1]==>[b,c]
        max_pool = max_pool.view([b,c])
        avg_pool = avg_pool.view([b,c])
 
        # 第一个全连接层下降通道数 [b,c]==>[b,c//4]
        x_maxpool = self.fc1(max_pool)
        x_avgpool = self.fc1(avg_pool)
 
        # 激活函数
        x_maxpool = self.leaky_relu(x_maxpool)
        x_avgpool = self.leaky_relu(x_avgpool)
        
        # 第二个全连接层恢复通道数 [b,c//4]==>[b,c]
        x_maxpool = self.fc2(x_maxpool)
        x_avgpool = self.fc2(x_avgpool)
        
        # 将这两种池化结果相加 [b,c]==>[b,c]
        x = x_maxpool + x_avgpool
        # sigmoid函数权值归一化
        x = self.sigmoid(x)
        # 调整维度 [b,c]==>[b,c,1,1]
        x = x.view([b,c,1,1])
        # 输入特征图和通道权重相乘 [b,c,h,w]
        outputs = inputs * x
        
        return outputs


class spatial_attention(nn.Module):
    def __init__(self, kernel_size=3):
        # 继承父类初始化方法
        super(spatial_attention, self).__init__()
        
        # 为了保持卷积前后的特征图shape相同，卷积时需要padding
        padding = kernel_size // 2
        # 7*7卷积融合通道信息 [b,2,h,w]==>[b,1,h,w]
        self.conv = nn.Conv2d(in_channels=2, out_channels=1, kernel_size=kernel_size, padding=padding, bias=False)
        # sigmoid函数
        self.sigmoid = nn.Sigmoid()
    
    # 前向传播
    def forward(self, inputs):
        
        # 在通道维度上最大池化 [b,1,h,w]  keepdim保留原有深度
        # 返回值是在某维度的最大值和对应的索引
        x_maxpool, _ = torch.max(inputs, dim=1, keepdim=True)
        #print(x_maxpool.shape)
        # 在通道维度上平均池化 [b,1,h,w]
        x_avgpool = torch.mean(inputs, dim=1, keepdim=True)
        # 池化后的结果在通道维度上堆叠 [b,2,h,w]
        x = torch.cat([x_maxpool, x_avgpool], dim=1)
        #print(x_maxpool.shape)
        # 卷积融合通道信息 [b,2,h,w]==>[b,1,h,w]
        x = self.conv(x)
        # 空间权重归一化
        x = self.sigmoid(x)
        # 输入特征图和空间权重相乘
       # print(inputs.shape) 
        #print(x.shape)
        outputs = inputs * x
        
        return outputs
    
    

class cbam(nn.Module):
    # 初始化，in_channel和ratio=4代表通道注意力机制的输入通道数和第一个全连接下降的通道数
    # kernel_size代表空间注意力机制的卷积核大小
    def __init__(self, in_channel, ratio=4, kernel_size=3):
        # 继承父类初始化方法
        super(cbam, self).__init__()
        
        # 实例化通道注意力机制
        self.channel_attention = channel_attention(in_channel=in_channel, ratio=ratio)
        # 实例化空间注意力机制
        self.spatial_attention = spatial_attention(kernel_size=kernel_size)
        self.avg = nn.AvgPool2d(2,stride=2,padding=0)
        
    # 前向传播
    def forward(self, inputs):
        
        # 先将输入图像经过通道注意力机制
        x1 = self.channel_attention(inputs)
        # 然后经过空间注意力机制
        x2 = self.spatial_attention(inputs)
        x = torch.cat([x1, x2], dim=1)
        x = self.avg(x)
        return x

In [42]:
#resqnet2
Accbox1 = []
Totallossbox1 = []
Accbox2 = []
Totallossbox2 = []
rec1_sam_train = [[],[],[],[]]
pre1_sam_train = [[],[],[],[]]
rec1_sam_test = [[],[],[],[]]
pre1_sam_test = [[],[],[],[]]
f1 = [[],[]]
f2 = [[],[]]
f3 = [[],[]]
f4 = [[],[]]
weight1 = [[],[],[]]
weight1_gra = [[],[],[]]
epoch = 300

a = torch.randn(8,5,requires_grad=True)
k = torch.randn(8,requires_grad=True)
b = torch.tensor(np.random.normal(0.01,0.01, size=(5)))#+torch.Tensor(5).uniform_(-0.01,0.1), dtype=torch.float32)
#Net2 for cifar100
class Net1(torch.nn.Module):
    def __init__(self):
        super(Net1, self).__init__()
        self.cbam1 = cbam(in_channel=8)
        self.cbam2 = cbam(in_channel=16)
        self.model1 = nn.Sequential(
        nn.Conv2d(in_channels = 3,out_channels = 8,kernel_size = 3,stride = 1,padding = 'same',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(8,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
            
        self.model2 = nn.Sequential(
        nn.Conv2d(in_channels = 32,out_channels = 4,kernel_size = 3,stride = 2,padding = 'valid',bias = False),
        nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(4,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
            
        nn.Flatten(),
        )
        
        self.model3 = nn.Sequential(
        nn.Conv2d(in_channels = 8,out_channels = 16,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(16,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.model4 = nn.Sequential(
        nn.Conv2d(in_channels = 16,out_channels = 32,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(32,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        self.fc1 = nn.Linear(4,4)
        self.fc = nn.Linear(4,4)
        
        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
       #         nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
       #         nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm1d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
       #         nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.Linear):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
       #         nn.init.constant_(m.bias, 0)
        
    def forward(self, x):     
        x = self.model1(x)
       # print(x.shape)
        x = self.cbam1(x)+self.model3(x)
       # print(x.shape)
        x = self.cbam2(x)+self.model4(x)
       # print(x.shape)
        x = self.model2(x)
        #print(x.shape)
        
        x = gate_operation_res2(x,8,a,k)
       
        #print(x.shape)
        #print(np.shape(x))
        #print('归一化',x)
        x = x.to(torch.float32)
        
        #print('men',x)
        x = measurement(x)
        #print('measure',x)
        x = x.to(torch.float32)
        #print(x)
        #noise = torch.tensor(np.random.normal(0.01,0.01, size=(8,8)), dtype=torch.float32)
        #self.fc2.weight.data = self.fc2.weight.data+noise
        x = F.leaky_relu(self.fc1(x))
        output = self.fc(x)
        #output = output.to(torch.float32)
        return output

model = Net1()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
summary(model,input_size =(3,32,32), batch_size = 32)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.005, weight_decay = 0.005)

#打印模型及模型优化时的参数
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

#保存模型参数
#state_mnist = {'net':model.state_dict(), 'optimizer':optimizer.state_dict(), 'epoch':epoch}
#torch.save(state_mnist,'C:/Users/Administrator/Desktop/CNN-mnist.pth') 

def train(epoch,train_loader,trainset_num):
    eps = 1e-8
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    TP_sam4_train, FP_sam4_train, FN_sam4_train = 0, 0, 0
    global Accbox1
    global Totallossbox1
    global rec1_sam_train 
    global pre1_sam_train
    global f1
    global f2
    global f3
    global f4
    global weight1
    old_time_train = time.time()

    for batch_idx, data in enumerate(train_loader,0):
        #print(batch_idx) #每一轮iteration，batch_idx从0到1874
        inputs, target = data
        #noisedata = add_noise(inputs)
        #print(inputs.shape)
        #print(target.shape)
        #print()
        #print(data)
        inputs, target = inputs.to(device),target.to(device)  #inputs为batchsize*3*32*32的向量，target为batchsize*1的向量，代表数字几
        #noisedata = noisedata.to(device)
        #print(inputs)
        target = target.type(torch.LongTensor)
        #print(target)
        #print()
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.requires_grad_(True)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print(outputs.shape)
        #print(outputs)
        predict = torch.argmax(outputs.data, dim=1)
        predict = predict.type(torch.FloatTensor)
        
        #print(predict)
        #print(predict.shape)
        #print(target.shape)
        #target.requires_grad_(True)
        #predict.requires_grad_(True)
        Total += target.size(0) #每个iteration中labels.size(0)为batchsize数
        perfect += (predict == target).sum().item()
        totalloss1 += loss.item()
    print("total loss in training set")
    print(totalloss1/len(train_loader))
    Totallossbox1 = np.append(Totallossbox1,totalloss1/len(train_loader))
            
    print('Accuracy on training set: %.3f %%' % (100 * perfect / Total))
    Accbox1 = np.append(Accbox1, perfect / Total)
    current_time_train = time.time()
    print("本次训练集运行时间为" + str(current_time_train - old_time_train) + "s")
    for i in range(len(target)):
        if predict[i] == 0 and target[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and target[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and target[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and target[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and target[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and target[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and target[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and target[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and target[i] == 2:
            FN_sam3_train += 1

        if predict[i] == 3 and target[i] == 3:
            TP_sam4_train += 1
        if predict[i] == 3 and target[i] != 3:
            FP_sam4_train += 1
        if predict[i] != 3 and target[i] == 3:
            FN_sam4_train += 1
            
    #计算四种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec1_sam_train[0] = np.append(rec1_sam_train[0],P1)
    pre1_sam_train[0] = np.append(pre1_sam_train[0],R1)
    f1[0] = np.append(f1[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec1_sam_train[1] = np.append(rec1_sam_train[1],P2)
    pre1_sam_train[1] = np.append(pre1_sam_train[1],R2)
    f2[0] = np.append(f2[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec1_sam_train[2] = np.append(rec1_sam_train[2],P3)
    pre1_sam_train[2] = np.append(pre1_sam_train[2],R3)
    f3[0] = np.append(f3[0],F1_3train)
    
    P4 = TP_sam4_train/(TP_sam4_train+FP_sam4_train+eps)
    R4 = TP_sam4_train/(TP_sam4_train+FN_sam4_train+eps)
    F1_4train = 2*P4*R4/(P4+R4+eps)
    rec1_sam_train[3] = np.append(rec1_sam_train[3],P4)
    pre1_sam_train[3] = np.append(pre1_sam_train[3],R4)
    f4[0] = np.append(f4[0],F1_4train)

    
def test(test_loader,testset_num):
    eps = 1e-8
    correct = 0
    total = 0
    totalloss2 = 0.0
    lab = []
    out = []
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    TP_sam4_test, FP_sam4_test, FN_sam4_test = 0, 0, 0
    global Accbox2
    global Totallossbox2
    global rec1_sam_test 
    global pre1_sam_test
    global f1
    global f2
    global f3
    global f4
    
    with torch.no_grad():  # 测试不需要计算梯度
        old_time_test = time.time()
        for data in test_loader:
            images, labels = data
           # noisedata = add_noise(images)
            images, labels = images.to(device),labels.to(device)
           # noisedata = noisedata.to(device)
            outputs = model(images)
            labels = labels.type(torch.LongTensor)
            loss = criterion(outputs,labels)
            lab = np.append(lab,labels.numpy())
            out = np.append(out,F.softmax(outputs).numpy())
            predicted = torch.argmax(outputs.data, dim=1)
            #此函数输出两个tensor,第一个tensor是每行的最大概率,第二个tensor是每行最大概率的索引,由于我们不需要获取最大概率的值，只要知道最大概率的是哪个类别即可,因此，我们只需要获取第二个tensor
            #print(predicted) #batchsize*1，均是0-9的数字
            #predicted = predicted.type(torch.FloatTensor)
            #labels = labels.type(torch.FloatTensor)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            #print(correct)
            totalloss2 += loss.item()
        Totallossbox2 = np.append(Totallossbox2,totalloss2/len(test_loader))
        print("total loss in testing set")
        print(totalloss2/len(test_loader))
        #totalloss2 = 0.0       
        print('Accuracy on test set: %.3f %%' % (100 * correct / total))
        Accbox2 = np.append(Accbox2, correct / total)
        current_time_test = time.time()
        print("本次测试集运行时间为" + str(current_time_test - old_time_test) + "s")
        print()
        print()
        print()
        print()
        for i in range(len(labels)):
            if predicted[i] == 0 and labels[i] == 0:
                TP_sam1_test += 1
            if predicted[i] == 0 and labels[i] != 0:
                FP_sam1_test += 1
            if predicted[i] != 0 and labels[i] == 0:
                FN_sam1_test += 1

            if predicted[i] == 1 and labels[i] == 1:
                TP_sam2_test += 1
            if predicted[i] == 1 and labels[i] != 1:
                FP_sam2_test += 1
            if predicted[i] != 1 and labels[i] == 1:
                FN_sam2_test += 1    

            if predicted[i] == 2 and labels[i] == 2:
                TP_sam3_test += 1
            if predicted[i] == 2 and labels[i] != 2:
                FP_sam3_test += 1
            if predicted[i] != 2 and labels[i] == 2:
                FN_sam3_test += 1

            if predicted[i] == 3 and labels[i] == 3:
                TP_sam4_test += 1
            if predicted[i] == 3 and labels[i] != 3:
                FP_sam4_test += 1
            if predicted[i] != 3 and labels[i] == 3:
                FN_sam4_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec1_sam_test[0] = np.append(rec1_sam_test[0],P1)
    pre1_sam_test[0] = np.append(pre1_sam_test[0],R1)
    f1[1] = np.append(f1[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec1_sam_test[1] = np.append(rec1_sam_test[1],P2)
    pre1_sam_test[1] = np.append(pre1_sam_test[1],R2)
    f2[1] = np.append(f2[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec1_sam_test[2] = np.append(rec1_sam_test[2],P3)
    pre1_sam_test[2] = np.append(pre1_sam_test[2],R3)
    f3[1] = np.append(f3[1],F1_3test)
    
    P4 = TP_sam4_test/(TP_sam4_test+FP_sam4_test+eps)
    R4 = TP_sam4_test/(TP_sam4_test+FN_sam4_test+eps)
    F1_4test = 2*P4*R4/(P4+R4+eps)
    rec1_sam_test[3] = np.append(rec1_sam_test[3],P4)
    pre1_sam_test[3] = np.append(pre1_sam_test[3],R4)
    f4[1] = np.append(f4[1],F1_4test)
    return lab,out
    
if __name__ == '__main__':
    for i in range(epoch):
        print('epoch:',i+1)
        #b = torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
        train(epoch,train_loader,2000)
        weight1[0] = np.append(weight1[0],a[0][0].item())
        weight1[1] = np.append(weight1[1],a[0][1].item())
        weight1[2] = np.append(weight1[2],a[0][2].item())
        if i != 0:
            weight1_gra[0] = np.append(weight1_gra[0],a.grad[0][0].item())
            weight1_gra[1] = np.append(weight1_gra[1],a.grad[0][1].item())
            weight1_gra[2] = np.append(weight1_gra[2],a.grad[0][2].item())
        label_0,output_0 = test(test_loader,400)
    output_0 = output_0.reshape(-1,4)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [32, 8, 32, 32]             216
       BatchNorm2d-2            [32, 8, 32, 32]              16
         LeakyReLU-3            [32, 8, 32, 32]               0
 AdaptiveMaxPool2d-4              [32, 8, 1, 1]               0
 AdaptiveAvgPool2d-5              [32, 8, 1, 1]               0
            Linear-6                    [32, 2]              16
            Linear-7                    [32, 2]              16
         LeakyReLU-8                    [32, 2]               0
         LeakyReLU-9                    [32, 2]               0
           Linear-10                    [32, 8]              16
           Linear-11                    [32, 8]              16
          Sigmoid-12                    [32, 8]               0
channel_attention-13            [32, 8, 32, 32]               0
           Conv2d-14            [32, 1,

C:\Users\15850\AppData\Local\Temp\ipykernel_3296\4208560114.py:274: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out = np.append(out,F.softmax(outputs).numpy())


total loss in testing set
1.2136623034110436
Accuracy on test set: 42.250 %
本次测试集运行时间为0.6973073482513428s




epoch: 2
total loss in training set
1.1493144971983773
Accuracy on training set: 45.000 %
本次训练集运行时间为7.496469974517822s
total loss in testing set
1.1189991785929754
Accuracy on test set: 52.500 %
本次测试集运行时间为0.871147632598877s




epoch: 3
total loss in training set
1.024084382587009
Accuracy on training set: 55.850 %
本次训练集运行时间为8.383383750915527s
total loss in testing set
0.9890474493686969
Accuracy on test set: 58.000 %
本次测试集运行时间为0.8794448375701904s




epoch: 4
total loss in training set
0.9293583574749175
Accuracy on training set: 59.100 %
本次训练集运行时间为9.962046146392822s
total loss in testing set
0.9276322447336637
Accuracy on test set: 62.500 %
本次测试集运行时间为0.5877809524536133s




epoch: 5
total loss in training set
0.88129432806893
Accuracy on training set: 62.500 %
本次训练集运行时间为7.1687095165252686s
total loss in testing set
0.864444796855633
Accuracy on test set: 65.250 %
本次测试集运行时间为0.

total loss in testing set
0.6885118530346797
Accuracy on test set: 74.500 %
本次测试集运行时间为0.5426421165466309s




epoch: 38
total loss in training set
0.5640694077998872
Accuracy on training set: 79.100 %
本次训练集运行时间为5.890120267868042s
total loss in testing set
0.6560724308857551
Accuracy on test set: 73.000 %
本次测试集运行时间为0.5649423599243164s




epoch: 39
total loss in training set
0.5777527564574801
Accuracy on training set: 78.450 %
本次训练集运行时间为6.007094621658325s
total loss in testing set
0.6716307195333334
Accuracy on test set: 74.750 %
本次测试集运行时间为0.6950218677520752s




epoch: 40
total loss in training set
0.5412855891008226
Accuracy on training set: 79.450 %
本次训练集运行时间为6.379284620285034s
total loss in testing set
0.6414310152714069
Accuracy on test set: 76.250 %
本次测试集运行时间为0.6503410339355469s




epoch: 41
total loss in training set
0.5507137957546446
Accuracy on training set: 79.350 %
本次训练集运行时间为6.263331413269043s
total loss in testing set
0.7270996708136338
Accuracy on test set: 71.750 %
本次测试

total loss in testing set
0.6104036867618561
Accuracy on test set: 74.500 %
本次测试集运行时间为0.5754127502441406s




epoch: 74
total loss in training set
0.4869920043718247
Accuracy on training set: 82.200 %
本次训练集运行时间为5.76896071434021s
total loss in testing set
0.6152519537852361
Accuracy on test set: 78.500 %
本次测试集运行时间为0.5531895160675049s




epoch: 75
total loss in training set
0.5004771115287902
Accuracy on training set: 81.250 %
本次训练集运行时间为5.844066381454468s
total loss in testing set
0.6246802050333756
Accuracy on test set: 78.500 %
本次测试集运行时间为0.5433714389801025s




epoch: 76
total loss in training set
0.5162401137843965
Accuracy on training set: 80.650 %
本次训练集运行时间为6.482088327407837s
total loss in testing set
0.5818969928301297
Accuracy on test set: 77.750 %
本次测试集运行时间为0.5136265754699707s




epoch: 77
total loss in training set
0.511862378744852
Accuracy on training set: 81.500 %
本次训练集运行时间为5.48363995552063s
total loss in testing set
0.6571035912403693
Accuracy on test set: 74.000 %
本次测试集运行

...

total loss in testing set
0.6693471601376166
Accuracy on test set: 75.250 %
本次测试集运行时间为0.48867082595825195s




epoch: 145
total loss in training set
0.4342121361266999
Accuracy on training set: 83.800 %
本次训练集运行时间为5.675991535186768s
total loss in testing set
0.6052074386523321
Accuracy on test set: 76.500 %
本次测试集运行时间为0.4792177677154541s




epoch: 146
total loss in training set
0.407852513686059
Accuracy on training set: 85.300 %
本次训练集运行时间为5.867169380187988s
total loss in testing set
0.5877934602590708
Accuracy on test set: 77.750 %
本次测试集运行时间为0.44200873374938965s




epoch: 147
total loss in training set
0.40975924924252527
Accuracy on training set: 85.100 %
本次训练集运行时间为5.032469987869263s
total loss in testing set
0.595119487780791
Accuracy on test set: 78.750 %
本次测试集运行时间为0.4579463005065918s




epoch: 148
total loss in training set
0.3984325629851175
Accuracy on training set: 86.750 %
本次训练集运行时间为4.950185298919678s
total loss in testing set
0.573417393060831
Accuracy on test set: 77.750 %


total loss in training set
0.3964338004589081
Accuracy on training set: 85.150 %
本次训练集运行时间为4.9921441078186035s
total loss in testing set
0.694971522459617
Accuracy on test set: 74.750 %
本次测试集运行时间为0.46886539459228516s




epoch: 181
total loss in training set
0.3857209055669724
Accuracy on training set: 86.000 %
本次训练集运行时间为4.96259069442749s
total loss in testing set
0.6403432694765238
Accuracy on test set: 75.250 %
本次测试集运行时间为0.4770023822784424s




epoch: 182
total loss in training set
0.40517581739122904
Accuracy on training set: 85.650 %
本次训练集运行时间为4.9436609745025635s
total loss in testing set
0.655279645553002
Accuracy on test set: 76.250 %
本次测试集运行时间为0.4419999122619629s




epoch: 183
total loss in training set
0.3752816842188911
Accuracy on training set: 87.350 %
本次训练集运行时间为4.9169652462005615s
total loss in testing set
0.6449126899242401
Accuracy on test set: 77.750 %
本次测试集运行时间为0.45676445960998535s




epoch: 184
total loss in training set
0.42838618845220594
Accuracy on training set: 

total loss in testing set
0.6182134334857647
Accuracy on test set: 78.500 %
本次测试集运行时间为0.49300360679626465s




epoch: 216
total loss in training set
0.3696394744846556
Accuracy on training set: 86.800 %
本次训练集运行时间为4.897900342941284s
total loss in testing set
0.6575711438289056
Accuracy on test set: 75.500 %
本次测试集运行时间为0.4459953308105469s




epoch: 217
total loss in training set
0.37463533807368504
Accuracy on training set: 86.450 %
本次训练集运行时间为5.017375469207764s
total loss in testing set
0.6486247732089117
Accuracy on test set: 76.750 %
本次测试集运行时间为0.45400118827819824s




epoch: 218
total loss in training set
0.3564720546442365
Accuracy on training set: 87.000 %
本次训练集运行时间为4.984508752822876s
total loss in testing set
0.6801773149233598
Accuracy on test set: 76.750 %
本次测试集运行时间为0.44899582862854004s




epoch: 219
total loss in training set
0.3458741550880765
Accuracy on training set: 87.450 %
本次训练集运行时间为4.891416311264038s
total loss in testing set
0.669501387155973
Accuracy on test set: 73.750

total loss in training set
0.3873408888540571
Accuracy on training set: 85.650 %
本次训练集运行时间为5.2741663455963135s
total loss in testing set
0.6836173992890578
Accuracy on test set: 76.000 %
本次测试集运行时间为0.5580086708068848s




epoch: 252
total loss in training set
0.3840787039389686
Accuracy on training set: 86.000 %
本次训练集运行时间为5.158014535903931s
total loss in testing set
0.7026604803708884
Accuracy on test set: 75.500 %
本次测试集运行时间为0.49187493324279785s




epoch: 253
total loss in training set
0.3598452385455843
Accuracy on training set: 88.000 %
本次训练集运行时间为4.917969703674316s
total loss in testing set
0.644585655285762
Accuracy on test set: 78.250 %
本次测试集运行时间为0.45999789237976074s




epoch: 254
total loss in training set
0.35902632985796246
Accuracy on training set: 87.100 %
本次训练集运行时间为4.9939751625061035s
total loss in testing set
0.7087802245066717
Accuracy on test set: 75.250 %
本次测试集运行时间为0.46500635147094727s




epoch: 255
total loss in training set
0.36320584468425265
Accuracy on training set

total loss in testing set
0.6912770087902362
Accuracy on test set: 74.750 %
本次测试集运行时间为0.4329972267150879s




epoch: 287
total loss in training set
0.3686737751676923
Accuracy on training set: 86.600 %
本次训练集运行时间为4.904447317123413s
total loss in testing set
0.7002230229286047
Accuracy on test set: 75.500 %
本次测试集运行时间为0.47800707817077637s




epoch: 288
total loss in training set
0.35370973876071354
Accuracy on training set: 86.700 %
本次训练集运行时间为4.925658464431763s
total loss in testing set
0.6920434534549713
Accuracy on test set: 74.500 %
本次测试集运行时间为0.4471249580383301s




epoch: 289
total loss in training set
0.3840591474657967
Accuracy on training set: 85.950 %
本次训练集运行时间为5.003209829330444s
total loss in testing set
0.7015420015041645
Accuracy on test set: 74.250 %
本次测试集运行时间为0.4340054988861084s




epoch: 290
total loss in training set
0.32319039863253396
Accuracy on training set: 88.250 %
本次训练集运行时间为5.018486976623535s
total loss in testing set
0.7204886629031255
Accuracy on test set: 73.000

In [9]:
# 将label值改为二分类以便画P-R图
label_1 = cp.deepcopy(label_0) 
label_2 = cp.deepcopy(label_0) 
label_3 = cp.deepcopy(label_0)

for i in range(len(label_0)):
    if label_0[i] != 0:
        label_0[i] = 4   
    else:
        label_0[i] = 0 
    if label_1[i] != 1:
        label_1[i] = 4 
    else:
        label_1[i] = 1 
    if label_2[i] != 2:
        label_2[i] = 4 
    else:
        label_2[i] = 2 
    if label_3[i] != 3:
        label_3[i] = 4 
    else:
        label_3[i] = 3
                      
y_true0 = np.array(label_0)
y_scores = np.array(output_0[:,0])
precision0, recall0, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr0, tpr0, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_1)
y_scores = np.array(output_0[:,1])
precision1, recall1, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr1, tpr1, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_2)
y_scores = np.array(output_0[:,2])
precision2, recall2, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr2, tpr2, thersholds = roc_curve(y_true2, y_scores, pos_label=2)

y_true3 = np.array(label_3)
y_scores = np.array(output_0[:,3])
precision3, recall3, thresholds = precision_recall_curve(y_true3, y_scores,pos_label=3)
fpr3, tpr3, thersholds = roc_curve(y_true3, y_scores, pos_label=3)

pr_auc0,pr_auc1,pr_auc2,pr_auc3 = auc(recall0,precision0),auc(recall1,precision1),auc(recall2,precision2),auc(recall3,precision3)
roc_auc0,roc_auc1,roc_auc2,roc_auc3 = auc(fpr0,tpr0),auc(fpr1,tpr1),auc(fpr2,tpr2),auc(fpr3,tpr3)


In [43]:
#QRCNN for 
Accbox11 = []
Totallossbox11 = []
Accbox22 = []
Totallossbox22 = []
rec11_sam_train = [[],[],[],[]]
pre11_sam_train = [[],[],[],[]]
rec11_sam_test = [[],[],[],[]]
pre11_sam_test = [[],[],[],[]]
f11 = [[],[]]
f22 = [[],[]]
f33 = [[],[]]
f44 = [[],[]]
weight11 = [[],[],[]]
weight11_gra = [[],[],[]]

#Net2 for mnist and fashionmnist
#mnist
class Net2(torch.nn.Module):
    def __init__(self):
        super(Net2, self).__init__()
        self.cbam1 = cbam(in_channel=8)
        self.cbam2 = cbam(in_channel=16)
        self.model1 = nn.Sequential(
        nn.Conv2d(in_channels = 3,out_channels = 8,kernel_size = 3,stride = 1,padding = 'same',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(8,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
            
        self.model2 = nn.Sequential(
        nn.Conv2d(in_channels = 32,out_channels = 4,kernel_size = 3,stride = 2,padding = 'valid',bias = False),
        nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(4,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
            
        nn.Flatten(),
        )
        
        self.model3 = nn.Sequential(
        nn.Conv2d(in_channels = 8,out_channels = 16,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(16,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.model4 = nn.Sequential(
        nn.Conv2d(in_channels = 16,out_channels = 32,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(32,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.fc1 = nn.Linear(4,4)
        self.fc2 = nn.Linear(12,12)
        self.fc = nn.Linear(4,4)

        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm1d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.Linear):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
          #      nn.init.constant_(m.bias, 0)
        
    def forward(self, x):
        x = self.model1(x)
       # print(x.shape)
        x = self.cbam1(x)+self.model3(x)
       # print(x.shape)
        x = self.cbam2(x)+self.model4(x)
       # print(x.shape)
        x = self.model2(x)
        x = x.to(torch.float32)
        #x = F.leaky_relu(self.fc1(x),0.5)
        #noise = torch.Tensor(5).uniform_(-0.01,0.1)
        #self.fc2.weight.data[0][0] = self.fc2.weight.data[0][0]+noise[0]
        #self.fc2.weight.data[0][1] = self.fc2.weight.data[0][1]+noise[1]
        #self.fc2.weight.data[0][2] = self.fc2.weight.data[0][2]+noise[2]
        #self.fc2.weight.data[0][3] = self.fc2.weight.data[0][3]+noise[3]
        #self.fc2.weight.data[0][4] = self.fc2.weight.data[0][4]+noise[4]
        #x = F.leaky_relu(self.fc2(x),0.5)
        #x = F.leaky_relu(self.fc2(x),0.5)
        #x = F.leaky_relu(self.fc2(x),0.5)
        x = gate_operation_res(x,8,a,k)
        x = F.leaky_relu(self.fc1(x),0.5)
        output = self.fc(x)
        return output

model = Net2()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
summary(model,input_size =(3,32,32), batch_size = 32)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.005, weight_decay = 0.005)

#打印模型及模型优化时的参数
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

#保存模型参数
#state_mnist = {'net':model.state_dict(), 'optimizer':optimizer.state_dict(), 'epoch':epoch}
#torch.save(state_mnist,'C:/Users/Administrator/Desktop/CNN-mnist.pth')

#def add_noise(inputs):
  #  noisy = inputs+torch.tensor(np.random.normal(0,1, size=( 3,32,32)), dtype=torch.float32)
  #  return noisy 

def train(epoch,train_loader,trainset_num):
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    eps = 1e-8
    global Accbox11
    global Totallossbox11
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    TP_sam4_train, FP_sam4_train, FN_sam4_train = 0, 0, 0
    global rec11_sam_train 
    global pre11_sam_train
    global f11
    global f22
    global f33
    global f44
    old_time_train = time.time()

    for batch_idx, data in enumerate(train_loader, 0):
        #print(batch_idx) #每一轮iteration，batch_idx从0到1874
        inputs, target = data
        #noisedata = add_noise(inputs)
        #print(inputs.shape)
        #print(target.shape)
        #print()
        #print(data)
        inputs, target = inputs.to(device),target.to(device)  #inputs为batchsize*1*28*28的向量，target为batchsize*1的向量，代表数字几
        #noisedata = noisedata.to(device)
        #print(inputs)
        #print(target)
        #print()
        outputs = model(inputs)
        target = target.type(torch.LongTensor)
        loss = criterion(outputs, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print(outputs.shape)
        #print(outputs)
        predict = torch.argmax(outputs.data, dim=1)
        #predict = predict.type(torch.FloatTensor)
        #target = target.type(torch.FloatTensor)
        #print(predict)
        #print(predict.shape)
        #print(target.shape)
        #target.requires_grad_(True)
        #predict.requires_grad_(True)
        Total += target.size(0) #每个iteration中labels.size(0)为batchsize数
        perfect += (predict == target).sum().item()
        totalloss1 += loss.item()
    print("total loss in training set")
    print(totalloss1/len(train_loader))
    Totallossbox11 = np.append(Totallossbox11,totalloss1/len(train_loader))
            
    print('Accuracy on training set: %.3f %%' % (100 * perfect / Total))
    Accbox11 = np.append(Accbox11, perfect / Total)
    current_time_train = time.time()
    print("本次训练集运行时间为" + str(current_time_train - old_time_train) + "s")
    for i in range(len(target)):
        if predict[i] == 0 and target[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and target[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and target[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and target[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and target[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and target[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and target[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and target[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and target[i] == 2:
            FN_sam3_train += 1

        if predict[i] == 3 and target[i] == 3:
            TP_sam4_train += 1
        if predict[i] == 3 and target[i] != 3:
            FP_sam4_train += 1
        if predict[i] != 3 and target[i] == 3:
            FN_sam4_train += 1
            
    #计算四种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec11_sam_train[0] = np.append(rec11_sam_train[0],P1)
    pre11_sam_train[0] = np.append(pre11_sam_train[0],R1)
    f11[0] = np.append(f11[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec11_sam_train[1] = np.append(rec11_sam_train[1],P2)
    pre11_sam_train[1] = np.append(pre11_sam_train[1],R2)
    f22[0] = np.append(f22[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec11_sam_train[2] = np.append(rec11_sam_train[2],P3)
    pre11_sam_train[2] = np.append(pre11_sam_train[2],R3)
    f33[0] = np.append(f33[0],F1_3train)
    
    P4 = TP_sam4_train/(TP_sam4_train+FP_sam4_train+eps)
    R4 = TP_sam4_train/(TP_sam4_train+FN_sam4_train+eps)
    F1_4train = 2*P4*R4/(P4+R4+eps)
    rec11_sam_train[3] = np.append(rec11_sam_train[3],P4)
    pre11_sam_train[3] = np.append(pre11_sam_train[3],R4)
    f44[0] = np.append(f44[0],F1_4train)

    
def test(test_loader,testset_num):
    correct = 0
    total = 0
    totalloss2 = 0.0
    eps = 1e-8
    lab = []
    out = []
    global Accbox22
    global Totallossbox22
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    TP_sam4_test, FP_sam4_test, FN_sam4_test = 0, 0, 0
    global rec11_sam_test 
    global pre11_sam_test
    global f11
    global f22
    global f33
    global f44
    
    with torch.no_grad():  # 测试不需要计算梯度
        old_time_test = time.time()
        for data in test_loader:
            images, labels = data
            #noisedata = add_noise(images)
            #print(images.shape) #batchsize*1*28*28
            #print(labels.shape) #batchsize*1
            images, labels = images.to(device),labels.to(device)
            #noisedata = noisedata.to(device)
            outputs = model(images)
            labels = labels.type(torch.LongTensor)
            loss = criterion(outputs,labels)
            lab = np.append(lab,labels.numpy())
            out = np.append(out,F.softmax(outputs).numpy())
            predicted = torch.argmax(outputs.data, dim=1)
            #此函数输出两个tensor,第一个tensor是每行的最大概率,第二个tensor是每行最大概率的索引,由于我们不需要获取最大概率的值，只要知道最大概率的是哪个类别即可,因此，我们只需要获取第二个tensor
            #print(predicted) #batchsize*1，均是0-9的数字
            #predicted = predicted.type(torch.FloatTensor)
            #labels = labels.type(torch.FloatTensor)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            #print(correct)
            totalloss2 += loss.item()
        Totallossbox22 = np.append(Totallossbox22,totalloss2/len(test_loader))
        print("total loss in testing set")
        print(totalloss2/len(test_loader))
        #totalloss2 = 0.0       
        print('Accuracy on test set: %.3f %%' % (100 * correct / total))
        Accbox22 = np.append(Accbox22, correct / total)
        current_time_test = time.time()
        print("本次测试集运行时间为" + str(current_time_test - old_time_test) + "s")
        print()
        print()
        print()
        for i in range(len(labels)):
            if predicted[i] == 0 and labels[i] == 0:
                TP_sam1_test += 1
            if predicted[i] == 0 and labels[i] != 0:
                FP_sam1_test += 1
            if predicted[i] != 0 and labels[i] == 0:
                FN_sam1_test += 1

            if predicted[i] == 1 and labels[i] == 1:
                TP_sam2_test += 1
            if predicted[i] == 1 and labels[i] != 1:
                FP_sam2_test += 1
            if predicted[i] != 1 and labels[i] == 1:
                FN_sam2_test += 1    

            if predicted[i] == 2 and labels[i] == 2:
                TP_sam3_test += 1
            if predicted[i] == 2 and labels[i] != 2:
                FP_sam3_test += 1
            if predicted[i] != 2 and labels[i] == 2:
                FN_sam3_test += 1

            if predicted[i] == 3 and labels[i] == 3:
                TP_sam4_test += 1
            if predicted[i] == 3 and labels[i] != 3:
                FP_sam4_test += 1
            if predicted[i] != 3 and labels[i] == 3:
                FN_sam4_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec11_sam_test[0] = np.append(rec11_sam_test[0],P1)
    pre11_sam_test[0] = np.append(pre11_sam_test[0],R1)
    f11[1] = np.append(f11[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec11_sam_test[1] = np.append(rec11_sam_test[1],P2)
    pre11_sam_test[1] = np.append(pre11_sam_test[1],R2)
    f22[1] = np.append(f22[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec11_sam_test[2] = np.append(rec11_sam_test[2],P3)
    pre11_sam_test[2] = np.append(pre11_sam_test[2],R3)
    f33[1] = np.append(f33[1],F1_3test)
    
    P4 = TP_sam4_test/(TP_sam4_test+FP_sam4_test+eps)
    R4 = TP_sam4_test/(TP_sam4_test+FN_sam4_test+eps)
    F1_4test = 2*P4*R4/(P4+R4+eps)
    rec11_sam_test[3] = np.append(rec11_sam_test[3],P4)
    pre11_sam_test[3] = np.append(pre11_sam_test[3],R4)
    f44[1] = np.append(f44[1],F1_4test)
    return lab,out   


if __name__ == '__main__':
    for i in range(epoch):
        print('epoch:',i+1)
        train(epoch,train_loader,2000)
        #b = torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
        weight11[0] = np.append(weight11[0],model.fc1.weight[0][0].item())
        weight11[1] = np.append(weight11[1],model.fc1.weight[0][1].item())
        weight11[2] = np.append(weight11[2],model.fc1.weight[0][2].item())
        if i != 0:
            weight11_gra[0] = np.append(weight11_gra[0],model.fc1.weight.grad[0][0].item())
            weight11_gra[1] = np.append(weight11_gra[1],model.fc1.weight.grad[0][1].item())
            weight11_gra[2] = np.append(weight11_gra[2],model.fc1.weight.grad[0][2].item())
        label_00,output_00 = test(test_loader,400)
    output_00 = output_00.reshape(-1,4)
    

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [32, 8, 32, 32]             216
       BatchNorm2d-2            [32, 8, 32, 32]              16
         LeakyReLU-3            [32, 8, 32, 32]               0
 AdaptiveMaxPool2d-4              [32, 8, 1, 1]               0
 AdaptiveAvgPool2d-5              [32, 8, 1, 1]               0
            Linear-6                    [32, 2]              16
            Linear-7                    [32, 2]              16
         LeakyReLU-8                    [32, 2]               0
         LeakyReLU-9                    [32, 2]               0
           Linear-10                    [32, 8]              16
           Linear-11                    [32, 8]              16
          Sigmoid-12                    [32, 8]               0
channel_attention-13            [32, 8, 32, 32]               0
           Conv2d-14            [32, 1,

C:\Users\15850\AppData\Local\Temp\ipykernel_3296\1994839929.py:272: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out = np.append(out,F.softmax(outputs).numpy())


total loss in testing set
1.2814597624998827
Accuracy on test set: 41.750 %
本次测试集运行时间为0.4796864986419678s



epoch: 2
total loss in training set
1.2266714478295946
Accuracy on training set: 45.650 %
本次训练集运行时间为5.126349925994873s
total loss in testing set
1.1491467494231005
Accuracy on test set: 51.000 %
本次测试集运行时间为0.46985578536987305s



epoch: 3
total loss in training set
1.13764749916773
Accuracy on training set: 51.550 %
本次训练集运行时间为4.978609085083008s
total loss in testing set
1.101116152910086
Accuracy on test set: 53.500 %
本次测试集运行时间为0.4640069007873535s



epoch: 4
total loss in training set
1.0346770267637948
Accuracy on training set: 58.300 %
本次训练集运行时间为4.9967474937438965s
total loss in testing set
0.9730692780934848
Accuracy on test set: 58.000 %
本次测试集运行时间为0.48470401763916016s



epoch: 5
total loss in training set
1.0024764641882882
Accuracy on training set: 58.850 %
本次训练集运行时间为5.0269622802734375s
total loss in testing set
0.9808826813331017
Accuracy on test set: 59.000 %
本次测试集运行时间为0

total loss in testing set
0.5887256287611448
Accuracy on test set: 73.500 %
本次测试集运行时间为0.4575459957122803s



epoch: 38
total loss in training set
0.5495302476107128
Accuracy on training set: 80.200 %
本次训练集运行时间为5.099515199661255s
total loss in testing set
0.6142529730613415
Accuracy on test set: 74.000 %
本次测试集运行时间为0.4440014362335205s



epoch: 39
total loss in training set
0.5232523003267864
Accuracy on training set: 79.700 %
本次训练集运行时间为4.943217039108276s
total loss in testing set
0.6743762355584365
Accuracy on test set: 75.750 %
本次测试集运行时间为0.4516890048980713s



epoch: 40
total loss in training set
0.52927651670244
Accuracy on training set: 80.400 %
本次训练集运行时间为5.07752799987793s
total loss in testing set
0.5723026348994329
Accuracy on test set: 79.750 %
本次测试集运行时间为0.5117974281311035s



epoch: 41
total loss in training set
0.507768366071913
Accuracy on training set: 80.750 %
本次训练集运行时间为4.9987382888793945s
total loss in testing set
0.6188253714488103
Accuracy on test set: 77.250 %
本次测试集运行时间为0

total loss in testing set
0.6023083604299105
Accuracy on test set: 78.750 %
本次测试集运行时间为0.5060849189758301s



epoch: 74
total loss in training set
0.44608614156170495
Accuracy on training set: 83.350 %
本次训练集运行时间为4.929730415344238s
total loss in testing set
0.66515890451578
Accuracy on test set: 77.500 %
本次测试集运行时间为0.46000027656555176s



epoch: 75
total loss in training set
0.43014925789265407
Accuracy on training set: 83.650 %
本次训练集运行时间为5.054082870483398s
total loss in testing set
0.5864584583502549
Accuracy on test set: 79.000 %
本次测试集运行时间为0.4645826816558838s



epoch: 76
total loss in training set
0.44979651485170635
Accuracy on training set: 83.300 %
本次训练集运行时间为5.0162036418914795s
total loss in testing set
0.6184036273222703
Accuracy on test set: 75.750 %
本次测试集运行时间为0.44283366203308105s



epoch: 77
total loss in training set
0.44612310945041594
Accuracy on training set: 83.200 %
本次训练集运行时间为4.987972021102905s
total loss in testing set
0.5792086055645576
Accuracy on test set: 77.250 %
本次测

...

total loss in training set
0.39583173607076916
Accuracy on training set: 84.800 %
本次训练集运行时间为5.242340564727783s
total loss in testing set
0.6680255004992852
Accuracy on test set: 75.500 %
本次测试集运行时间为0.4969966411590576s



epoch: 146
total loss in training set
0.405323987678876
Accuracy on training set: 84.500 %
本次训练集运行时间为4.928581237792969s
total loss in testing set
0.6506933203110328
Accuracy on test set: 75.750 %
本次测试集运行时间为0.466998815536499s



epoch: 147
total loss in training set
0.3946924753605373
Accuracy on training set: 85.600 %
本次训练集运行时间为4.982569932937622s
total loss in testing set
0.6384789645671844
Accuracy on test set: 79.000 %
本次测试集运行时间为0.45499753952026367s



epoch: 148
total loss in training set
0.36712659020272514
Accuracy on training set: 86.750 %
本次训练集运行时间为5.078850746154785s
total loss in testing set
0.6099997701553198
Accuracy on test set: 78.250 %
本次测试集运行时间为0.4790043830871582s



epoch: 149
total loss in training set
0.36701819608135827
Accuracy on training set: 86.450

total loss in training set
0.3492101516042437
Accuracy on training set: 86.800 %
本次训练集运行时间为5.045358419418335s
total loss in testing set
0.6125966402200552
Accuracy on test set: 78.250 %
本次测试集运行时间为0.46077394485473633s



epoch: 182
total loss in training set
0.37091256199138506
Accuracy on training set: 86.450 %
本次训练集运行时间为5.0442893505096436s
total loss in testing set
0.6162366477342752
Accuracy on test set: 75.750 %
本次测试集运行时间为0.49878787994384766s



epoch: 183
total loss in training set
0.3794726419543463
Accuracy on training set: 85.750 %
本次训练集运行时间为4.952392816543579s
total loss in testing set
0.6490481610481555
Accuracy on test set: 76.750 %
本次测试集运行时间为0.4529993534088135s



epoch: 184
total loss in training set
0.38601237819308326
Accuracy on training set: 86.300 %
本次训练集运行时间为4.883125066757202s
total loss in testing set
0.5923306598113134
Accuracy on test set: 77.750 %
本次测试集运行时间为0.47888970375061035s



epoch: 185
total loss in training set
0.36727544523420785
Accuracy on training set: 8

total loss in testing set
0.6362125185819772
Accuracy on test set: 76.750 %
本次测试集运行时间为0.4590010643005371s



epoch: 217
total loss in training set
0.34783056212796104
Accuracy on training set: 87.350 %
本次训练集运行时间为4.912038803100586s
total loss in testing set
0.6624001333346734
Accuracy on test set: 77.000 %
本次测试集运行时间为0.4670088291168213s



epoch: 218
total loss in training set
0.3635678083177597
Accuracy on training set: 86.100 %
本次训练集运行时间为5.038669586181641s
total loss in testing set
0.6797537597326132
Accuracy on test set: 76.250 %
本次测试集运行时间为0.4780006408691406s



epoch: 219
total loss in training set
0.368924408323235
Accuracy on training set: 86.300 %
本次训练集运行时间为4.920408010482788s
total loss in testing set
0.6024996775847214
Accuracy on test set: 79.500 %
本次测试集运行时间为0.47499656677246094s



epoch: 220
total loss in training set
0.3614269821416764
Accuracy on training set: 86.100 %
本次训练集运行时间为5.019535064697266s
total loss in testing set
0.6987102742378528
Accuracy on test set: 77.250 %
本次测

total loss in testing set
0.5334424101389371
Accuracy on test set: 80.250 %
本次测试集运行时间为0.497997522354126s



epoch: 253
total loss in training set
0.34951793248691254
Accuracy on training set: 87.400 %
本次训练集运行时间为4.974874496459961s
total loss in testing set
0.592766997905878
Accuracy on test set: 79.250 %
本次测试集运行时间为0.47246313095092773s



epoch: 254
total loss in training set
0.3430944527425463
Accuracy on training set: 86.900 %
本次训练集运行时间为5.133350849151611s
total loss in testing set
0.6866683799486893
Accuracy on test set: 76.000 %
本次测试集运行时间为0.49899888038635254s



epoch: 255
total loss in training set
0.3558760131635363
Accuracy on training set: 87.600 %
本次训练集运行时间为4.901707649230957s
total loss in testing set
0.5471072907631214
Accuracy on test set: 79.250 %
本次测试集运行时间为0.5239994525909424s



epoch: 256
total loss in training set
0.34505984139820883
Accuracy on training set: 87.200 %
本次训练集运行时间为4.996987581253052s
total loss in testing set
0.6478833097677964
Accuracy on test set: 78.500 %
本次

total loss in testing set
0.5832552726452167
Accuracy on test set: 78.250 %
本次测试集运行时间为0.4479994773864746s



epoch: 289
total loss in training set
0.33427925644412876
Accuracy on training set: 87.600 %
本次训练集运行时间为4.947605848312378s
total loss in testing set
0.665751936344
Accuracy on test set: 77.750 %
本次测试集运行时间为0.49899959564208984s



epoch: 290
total loss in training set
0.3209779892885496
Accuracy on training set: 88.650 %
本次训练集运行时间为4.911521911621094s
total loss in testing set
0.5878643955175693
Accuracy on test set: 79.750 %
本次测试集运行时间为0.47100162506103516s



epoch: 291
total loss in training set
0.3587578998671638
Accuracy on training set: 86.450 %
本次训练集运行时间为4.9593164920806885s
total loss in testing set
0.6403695413699517
Accuracy on test set: 79.000 %
本次测试集运行时间为0.49051570892333984s



epoch: 292
total loss in training set
0.3431237992786226
Accuracy on training set: 88.100 %
本次训练集运行时间为5.095625877380371s
total loss in testing set
0.7046661606201758
Accuracy on test set: 77.000 %
本次测

In [11]:
# 将label值改为二分类以便画P-R图
label_11 = cp.deepcopy(label_00) 
label_22 = cp.deepcopy(label_00) 
label_33 = cp.deepcopy(label_00)

for i in range(len(label_00)):
    if label_00[i] != 0:
        label_00[i] = 4   
    else:
        label_00[i] = 0 
    if label_11[i] != 1:
        label_11[i] = 4 
    else:
        label_11[i] = 1 
    if label_22[i] != 2:
        label_22[i] = 4 
    else:
        label_22[i] = 2 
    if label_33[i] != 3:
        label_33[i] = 4 
    else:
        label_33[i] = 3

y_true0 = np.array(label_00)
y_scores = np.array(output_00[:,0])
precision00, recall00, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr00, tpr00, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_11)
y_scores = np.array(output_00[:,1])
precision11, recall11, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr11, tpr11, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_22)
y_scores = np.array(output_00[:,2])
precision22, recall22, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr22, tpr22, thersholds = roc_curve(y_true2, y_scores, pos_label=2)

y_true3 = np.array(label_33)
y_scores = np.array(output_00[:,3])
precision33, recall33, thresholds = precision_recall_curve(y_true3, y_scores,pos_label=3)
fpr33, tpr33, thersholds = roc_curve(y_true3, y_scores, pos_label=3)

pr_auc00,pr_auc11,pr_auc22,pr_auc33 = auc(recall00,precision00),auc(recall11,precision11),auc(recall22,precision22),auc(recall33,precision33)
roc_auc00,roc_auc11,roc_auc22,roc_auc33 = auc(fpr00,tpr00),auc(fpr11,tpr11),auc(fpr22,tpr22),auc(fpr33,tpr33)


In [44]:
#QDCNN
Accbox111 = []
Totallossbox111 = []
Accbox222 = []
Totallossbox222 = []
rec111_sam_train = [[],[],[],[]]
pre111_sam_train = [[],[],[],[]]
rec111_sam_test = [[],[],[],[]]
pre111_sam_test = [[],[],[],[]]
f111 = [[],[]]
f222 = [[],[]]
f333 = [[],[]]
f444 = [[],[]]
weight111 = [[],[],[]]
weight111_gra = [[],[],[]]
 
c = torch.randn(8,5,requires_grad=True)
k1 = torch.randn(28,requires_grad=True)
b = torch.Tensor(5).uniform_(-0.01,0.1)+torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
#Net2 for mnist and fashionmnist
#mnist
class Net3(torch.nn.Module):
    def __init__(self):
        super(Net3, self).__init__()
        self.cbam1 = cbam(in_channel=8)
        self.cbam2 = cbam(in_channel=16)
        self.model1 = nn.Sequential(
        nn.Conv2d(in_channels = 3,out_channels = 8,kernel_size = 3,stride = 1,padding = 'same',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(8,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
            
        self.model2 = nn.Sequential(
        nn.Conv2d(in_channels = 32,out_channels = 4,kernel_size = 3,stride = 2,padding = 'valid',bias = False),
        nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(4,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
            
        nn.Flatten(),
        )
        
        self.model3 = nn.Sequential(
        nn.Conv2d(in_channels = 8,out_channels = 16,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(16,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.model4 = nn.Sequential(
        nn.Conv2d(in_channels = 16,out_channels = 32,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(32,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.model5 = nn.Sequential(
        nn.Conv2d(in_channels = 8,out_channels = 32,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        nn.MaxPool2d(kernel_size = 2),
        nn.BatchNorm2d(32,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.fc1 = nn.Linear(4,4)
        self.fc = nn.Linear(4,4)

        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm1d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.Linear):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
          #      nn.init.constant_(m.bias, 0)
        
    def forward(self, x):
        x = self.model1(x)
       # print(x.shape)
        x1 = self.model5(x)
        x = self.cbam1(x)+self.model3(x)
       # print(x.shape)
        x = self.cbam2(x)+self.model4(x)+x1
       # print(x.shape)
        x = self.model2(x)
       # print(x.shape)
        
        #x = self.cbam(x)
        
        x = gate_operation_dense(x,8,c,k1)
       
        #print(x.shape)
        #print(np.shape(x))
        #print('归一化',x)
        x = x.to(torch.float32)
        
        #print('men',x)
        x = measurement(x)
        #print('measure',x)
        x = x.to(torch.float32)
        #print(x)
        x = F.leaky_relu(self.fc1(x))
        output = self.fc(x)
        #output = output.to(torch.float32)
        #print(output.shape)
        #x = rotation(x,A=1,a=0.8,psi=math.pi/2)
        #output = self.model8(x)
        return output

model = Net3()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
summary(model,input_size =(3,32,32), batch_size = 32)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.005, weight_decay = 0.005)

#打印模型及模型优化时的参数
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

#保存模型参数
#state_mnist = {'net':model.state_dict(), 'optimizer':optimizer.state_dict(), 'epoch':epoch}
#torch.save(state_mnist,'C:/Users/Administrator/Desktop/CNN-mnist.pth')

#def add_noise(inputs):
#    noisy = inputs+torch.tensor(np.random.normal(0,1, size=( 3,32,32)), dtype=torch.float32)
#    return noisy 

def train(epoch,train_loader,trainset_num):
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    eps = 1e-8
    global Accbox111
    global Totallossbox111
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    TP_sam4_train, FP_sam4_train, FN_sam4_train = 0, 0, 0
    global rec111_sam_train 
    global pre111_sam_train
    global f111
    global f222
    global f333
    global f444
    old_time_train = time.time()

    for batch_idx, data in enumerate(train_loader, 0):
        #print(batch_idx) #每一轮iteration，batch_idx从0到1874
        inputs, target = data
       # noisedata = add_noise(inputs,0.5)
        #print(inputs.shape)
        #print(target.shape)
        #print()
        #print(data)
        inputs, target = inputs.to(device),target.to(device)  #inputs为batchsize*1*28*28的向量，target为batchsize*1的向量，代表数字几
       # noisedata = noisedata.to(device)
        #print(inputs)
        #print(target)
        #print()
        outputs = model(inputs)
        target = target.type(torch.LongTensor)
        loss = criterion(outputs, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print(outputs.shape)
        #print(outputs)
        predict = torch.argmax(outputs.data, dim=1)
        #predict = predict.type(torch.FloatTensor)
        #target = target.type(torch.FloatTensor)
        #print(predict)
        #print(predict.shape)
        #print(target.shape)
        #target.requires_grad_(True)
        #predict.requires_grad_(True)
        Total += target.size(0) #每个iteration中labels.size(0)为batchsize数
        perfect += (predict == target).sum().item()
        totalloss1 += loss.item()
    print("total loss in training set")
    print(totalloss1/len(train_loader))
    Totallossbox111 = np.append(Totallossbox111,totalloss1/len(train_loader))
            
    print('Accuracy on training set: %.3f %%' % (100 * perfect / Total))
    Accbox111 = np.append(Accbox111, perfect / Total)
    current_time_train = time.time()
    print("本次训练集运行时间为" + str(current_time_train - old_time_train) + "s")
    for i in range(len(target)):
        if predict[i] == 0 and target[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and target[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and target[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and target[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and target[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and target[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and target[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and target[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and target[i] == 2:
            FN_sam3_train += 1

        if predict[i] == 3 and target[i] == 3:
            TP_sam4_train += 1
        if predict[i] == 3 and target[i] != 3:
            FP_sam4_train += 1
        if predict[i] != 3 and target[i] == 3:
            FN_sam4_train += 1
            
    #计算四种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec111_sam_train[0] = np.append(rec111_sam_train[0],P1)
    pre111_sam_train[0] = np.append(pre111_sam_train[0],R1)
    f111[0] = np.append(f111[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec111_sam_train[1] = np.append(rec111_sam_train[1],P2)
    pre111_sam_train[1] = np.append(pre111_sam_train[1],R2)
    f222[0] = np.append(f222[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec111_sam_train[2] = np.append(rec111_sam_train[2],P3)
    pre111_sam_train[2] = np.append(pre111_sam_train[2],R3)
    f333[0] = np.append(f333[0],F1_3train)
    
    P4 = TP_sam4_train/(TP_sam4_train+FP_sam4_train+eps)
    R4 = TP_sam4_train/(TP_sam4_train+FN_sam4_train+eps)
    F1_4train = 2*P4*R4/(P4+R4+eps)
    rec111_sam_train[3] = np.append(rec111_sam_train[3],P4)
    pre111_sam_train[3] = np.append(pre111_sam_train[3],R4)
    f444[0] = np.append(f444[0],F1_4train)

    
def test(test_loader,testset_num):
    correct = 0
    total = 0
    totalloss2 = 0.0
    eps = 1e-8
    lab = []
    out = []
    global Accbox222
    global Totallossbox222
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    TP_sam4_test, FP_sam4_test, FN_sam4_test = 0, 0, 0
    global rec111_sam_test 
    global pre111_sam_test
    global f111
    global f222
    global f333
    global f444
    
    with torch.no_grad():  # 测试不需要计算梯度
        old_time_test = time.time()
        for data in test_loader:
            images, labels = data
            #noisedata = add_noise(images,0.2)
            #print(images.shape) #batchsize*1*28*28
            #print(labels.shape) #batchsize*1
            images, labels = images.to(device),labels.to(device)
            #noisedata = noisedata.to(device)
            outputs = model(images)
            labels = labels.type(torch.LongTensor)
            loss = criterion(outputs,labels)
            lab = np.append(lab,labels.numpy())
            out = np.append(out,F.softmax(outputs).numpy())
            predicted = torch.argmax(outputs.data, dim=1)
            #此函数输出两个tensor,第一个tensor是每行的最大概率,第二个tensor是每行最大概率的索引,由于我们不需要获取最大概率的值，只要知道最大概率的是哪个类别即可,因此，我们只需要获取第二个tensor
            #print(predicted) #batchsize*1，均是0-9的数字
            #predicted = predicted.type(torch.FloatTensor)
            #labels = labels.type(torch.FloatTensor)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            #print(correct)
            totalloss2 += loss.item()
        Totallossbox222 = np.append(Totallossbox222,totalloss2/len(test_loader))
        print("total loss in testing set")
        print(totalloss2/len(test_loader))
        #totalloss2 = 0.0       
        print('Accuracy on test set: %.3f %%' % (100 * correct / total))
        Accbox222 = np.append(Accbox222, correct / total)
        current_time_test = time.time()
        print("本次测试集运行时间为" + str(current_time_test - old_time_test) + "s")
        print()
        print()
        print()
        for i in range(len(labels)):
            if predicted[i] == 0 and labels[i] == 0:
                TP_sam1_test += 1
            if predicted[i] == 0 and labels[i] != 0:
                FP_sam1_test += 1
            if predicted[i] != 0 and labels[i] == 0:
                FN_sam1_test += 1

            if predicted[i] == 1 and labels[i] == 1:
                TP_sam2_test += 1
            if predicted[i] == 1 and labels[i] != 1:
                FP_sam2_test += 1
            if predicted[i] != 1 and labels[i] == 1:
                FN_sam2_test += 1    

            if predicted[i] == 2 and labels[i] == 2:
                TP_sam3_test += 1
            if predicted[i] == 2 and labels[i] != 2:
                FP_sam3_test += 1
            if predicted[i] != 2 and labels[i] == 2:
                FN_sam3_test += 1

            if predicted[i] == 3 and labels[i] == 3:
                TP_sam4_test += 1
            if predicted[i] == 3 and labels[i] != 3:
                FP_sam4_test += 1
            if predicted[i] != 3 and labels[i] == 3:
                FN_sam4_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec111_sam_test[0] = np.append(rec111_sam_test[0],P1)
    pre111_sam_test[0] = np.append(pre111_sam_test[0],R1)
    f111[1] = np.append(f111[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec111_sam_test[1] = np.append(rec111_sam_test[1],P2)
    pre111_sam_test[1] = np.append(pre111_sam_test[1],R2)
    f222[1] = np.append(f222[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec111_sam_test[2] = np.append(rec111_sam_test[2],P3)
    pre111_sam_test[2] = np.append(pre111_sam_test[2],R3)
    f333[1] = np.append(f333[1],F1_3test)

    P4 = TP_sam4_test/(TP_sam4_test+FP_sam4_test+eps)
    R4 = TP_sam4_test/(TP_sam4_test+FN_sam4_test+eps)
    F1_4test = 2*P4*R4/(P4+R4+eps)
    rec111_sam_test[3] = np.append(rec111_sam_test[3],P4)
    pre111_sam_test[3] = np.append(pre111_sam_test[3],R4)
    f444[1] = np.append(f444[1],F1_4test)
    return lab,out   


if __name__ == '__main__':
    for i in range(epoch):
        print('epoch:',i+1)
        #b = torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
        train(epoch,train_loader,2000)
        weight111[0] = np.append(weight111[0],c[0][0].item())
        weight111[1] = np.append(weight111[1],c[0][1].item())
        weight111[2] = np.append(weight111[2],c[0][2].item())
        if i != 0:
            weight111_gra[0] = np.append(weight111_gra[0],c.grad[0][0].item())
            weight111_gra[1] = np.append(weight111_gra[1],c.grad[0][1].item())
            weight111_gra[2] = np.append(weight111_gra[2],c.grad[0][2].item())
        label_000,output_000 = test(test_loader,400)
    output_000 = output_000.reshape(-1,4)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [32, 8, 32, 32]             216
       BatchNorm2d-2            [32, 8, 32, 32]              16
         LeakyReLU-3            [32, 8, 32, 32]               0
            Conv2d-4           [32, 32, 16, 16]             256
         MaxPool2d-5             [32, 32, 8, 8]               0
       BatchNorm2d-6             [32, 32, 8, 8]              64
         LeakyReLU-7             [32, 32, 8, 8]               0
 AdaptiveMaxPool2d-8              [32, 8, 1, 1]               0
 AdaptiveAvgPool2d-9              [32, 8, 1, 1]               0
           Linear-10                    [32, 2]              16
           Linear-11                    [32, 2]              16
        LeakyReLU-12                    [32, 2]               0
        LeakyReLU-13                    [32, 2]               0
           Linear-14                   

C:\Users\15850\AppData\Local\Temp\ipykernel_3296\2452077550.py:290: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out = np.append(out,F.softmax(outputs).numpy())


total loss in testing set
1.161993031318371
Accuracy on test set: 46.750 %
本次测试集运行时间为0.4940056800842285s



epoch: 2
total loss in training set
1.1360321745039925
Accuracy on training set: 47.000 %
本次训练集运行时间为5.500371694564819s
total loss in testing set
1.1113954782485962
Accuracy on test set: 47.500 %
本次测试集运行时间为0.4880049228668213s



epoch: 3
total loss in training set
1.0815156281940521
Accuracy on training set: 49.500 %
本次训练集运行时间为5.47453498840332s
total loss in testing set
1.0646502146354089
Accuracy on test set: 49.750 %
本次测试集运行时间为0.49599575996398926s



epoch: 4
total loss in training set
1.0506494102023898
Accuracy on training set: 53.400 %
本次训练集运行时间为5.513137340545654s
total loss in testing set
1.0419911604661207
Accuracy on test set: 52.250 %
本次测试集运行时间为0.507948637008667s



epoch: 5
total loss in training set
1.0269410042535692
Accuracy on training set: 54.000 %
本次训练集运行时间为5.491795778274536s
total loss in testing set
0.9885708368741549
Accuracy on test set: 56.500 %
本次测试集运行时间为0.49

total loss in testing set
0.7208773631315964
Accuracy on test set: 74.500 %
本次测试集运行时间为0.477003812789917s



epoch: 38
total loss in training set
0.4972818794231566
Accuracy on training set: 80.800 %
本次训练集运行时间为5.360576868057251s
total loss in testing set
0.7867466050844926
Accuracy on test set: 71.750 %
本次测试集运行时间为0.5050034523010254s



epoch: 39
total loss in training set
0.46903508475848604
Accuracy on training set: 81.200 %
本次训练集运行时间为5.449821949005127s
total loss in testing set
0.7929403644341689
Accuracy on test set: 72.000 %
本次测试集运行时间为0.482194185256958s



epoch: 40
total loss in training set
0.4687710028319132
Accuracy on training set: 82.350 %
本次训练集运行时间为5.383466958999634s
total loss in testing set
0.735842567223769
Accuracy on test set: 71.500 %
本次测试集运行时间为0.5079877376556396s



epoch: 41
total loss in training set
0.471239434111686
Accuracy on training set: 81.400 %
本次训练集运行时间为5.471857309341431s
total loss in testing set
0.7491133029644306
Accuracy on test set: 75.750 %
本次测试集运行时间为0

total loss in testing set
0.7305696308612823
Accuracy on test set: 73.000 %
本次测试集运行时间为0.521003246307373s



epoch: 74
total loss in training set
0.3699811188474534
Accuracy on training set: 86.600 %
本次训练集运行时间为5.483299493789673s
total loss in testing set
0.7550198458708249
Accuracy on test set: 74.250 %
本次测试集运行时间为0.5430076122283936s



epoch: 75
total loss in training set
0.3853011606704621
Accuracy on training set: 86.200 %
本次训练集运行时间为5.4473795890808105s
total loss in testing set
0.720194167815722
Accuracy on test set: 75.500 %
本次测试集运行时间为0.5080001354217529s



epoch: 76
total loss in training set
0.3527296968395748
Accuracy on training set: 86.500 %
本次训练集运行时间为5.405045986175537s
total loss in testing set
0.7271246703771445
Accuracy on test set: 75.250 %
本次测试集运行时间为0.48099780082702637s



epoch: 77
total loss in training set
0.39137050201968543
Accuracy on training set: 84.800 %
本次训练集运行时间为5.413654565811157s
total loss in testing set
0.7262701690196991
Accuracy on test set: 75.000 %
本次测试集运行

...

total loss in testing set
0.7926228596613958
Accuracy on test set: 74.250 %
本次测试集运行时间为0.4876120090484619s



epoch: 146
total loss in training set
0.32058941584730904
Accuracy on training set: 87.950 %
本次训练集运行时间为5.362741231918335s
total loss in testing set
0.8626096294476435
Accuracy on test set: 74.750 %
本次测试集运行时间为0.5170025825500488s



epoch: 147
total loss in training set
0.32299752734483234
Accuracy on training set: 88.450 %
本次训练集运行时间为5.6106836795806885s
total loss in testing set
1.066813968695127
Accuracy on test set: 71.250 %
本次测试集运行时间为0.5118520259857178s



epoch: 148
total loss in training set
0.330856093575084
Accuracy on training set: 87.300 %
本次训练集运行时间为5.491668224334717s
total loss in testing set
0.8805509301332327
Accuracy on test set: 76.000 %
本次测试集运行时间为0.5150010585784912s



epoch: 149
total loss in training set
0.31563619607024723
Accuracy on training set: 87.800 %
本次训练集运行时间为5.468591213226318s
total loss in testing set
0.8851814980690296
Accuracy on test set: 74.500 %
本次

total loss in testing set
0.7428955917175
Accuracy on test set: 75.000 %
本次测试集运行时间为0.5275945663452148s



epoch: 182
total loss in training set
0.29693679819031366
Accuracy on training set: 89.800 %
本次训练集运行时间为5.505418300628662s
total loss in testing set
0.8284938335418701
Accuracy on test set: 76.500 %
本次测试集运行时间为0.5249948501586914s



epoch: 183
total loss in training set
0.3429245184811335
Accuracy on training set: 87.150 %
本次训练集运行时间为5.365396022796631s
total loss in testing set
0.778972524863023
Accuracy on test set: 75.750 %
本次测试集运行时间为0.5455818176269531s



epoch: 184
total loss in training set
0.3363161087036133
Accuracy on training set: 86.300 %
本次训练集运行时间为5.474365949630737s
total loss in testing set
0.8465279455368335
Accuracy on test set: 75.000 %
本次测试集运行时间为0.5315639972686768s



epoch: 185
total loss in training set
0.3063035276201036
Accuracy on training set: 88.500 %
本次训练集运行时间为5.546006679534912s
total loss in testing set
0.8073827417997214
Accuracy on test set: 73.250 %
本次测试集运行

total loss in testing set
0.7971508182012118
Accuracy on test set: 76.000 %
本次测试集运行时间为0.503000020980835s



epoch: 218
total loss in training set
0.3647541668679979
Accuracy on training set: 87.050 %
本次训练集运行时间为5.5446367263793945s
total loss in testing set
0.8925333940065824
Accuracy on test set: 74.750 %
本次测试集运行时间为0.5196492671966553s



epoch: 219
total loss in training set
0.32414872151991675
Accuracy on training set: 88.200 %
本次训练集运行时间为5.504763841629028s
total loss in testing set
0.7904773996426508
Accuracy on test set: 75.750 %
本次测试集运行时间为0.45800113677978516s



epoch: 220
total loss in training set
0.31170979804462856
Accuracy on training set: 88.700 %
本次训练集运行时间为5.440306901931763s
total loss in testing set
0.7833539109963638
Accuracy on test set: 75.500 %
本次测试集运行时间为0.4909496307373047s



epoch: 221
total loss in training set
0.2950282200934395
Accuracy on training set: 90.400 %
本次训练集运行时间为5.45225715637207s
total loss in testing set
0.7573143885685847
Accuracy on test set: 76.500 %
本次

total loss in testing set
0.8332536793672122
Accuracy on test set: 73.750 %
本次测试集运行时间为0.46900296211242676s



epoch: 254
total loss in training set
0.3053618155065037
Accuracy on training set: 88.500 %
本次训练集运行时间为5.475840330123901s
total loss in testing set
0.7181156575679779
Accuracy on test set: 79.000 %
本次测试集运行时间为0.448833703994751s



epoch: 255
total loss in training set
0.34426886316329713
Accuracy on training set: 87.550 %
本次训练集运行时间为5.460693597793579s
total loss in testing set
0.8634500503540039
Accuracy on test set: 73.000 %
本次测试集运行时间为0.47600317001342773s



epoch: 256
total loss in training set
0.2815632022444218
Accuracy on training set: 90.050 %
本次训练集运行时间为5.467684268951416s
total loss in testing set
0.7948208313721877
Accuracy on test set: 76.250 %
本次测试集运行时间为0.5209996700286865s



epoch: 257
total loss in training set
0.25269244232821086
Accuracy on training set: 90.150 %
本次训练集运行时间为5.484987020492554s
total loss in testing set
0.8795481094947228
Accuracy on test set: 74.250 %
本

total loss in testing set
0.7935258104250982
Accuracy on test set: 77.750 %
本次测试集运行时间为0.5178353786468506s



epoch: 290
total loss in training set
0.3063040900798071
Accuracy on training set: 88.800 %
本次训练集运行时间为5.431658983230591s
total loss in testing set
0.8217976941512182
Accuracy on test set: 78.000 %
本次测试集运行时间为0.5510036945343018s



epoch: 291
total loss in training set
0.29784383305481504
Accuracy on training set: 89.100 %
本次训练集运行时间为5.448946952819824s
total loss in testing set
0.8446722443287189
Accuracy on test set: 75.750 %
本次测试集运行时间为0.49500417709350586s



epoch: 292
total loss in training set
0.2997636325539105
Accuracy on training set: 89.350 %
本次训练集运行时间为5.391040086746216s
total loss in testing set
0.8355336922865647
Accuracy on test set: 74.750 %
本次测试集运行时间为0.4770021438598633s



epoch: 293
total loss in training set
0.259862753015662
Accuracy on training set: 90.450 %
本次训练集运行时间为5.62227463722229s
total loss in testing set
0.7850839518583738
Accuracy on test set: 76.500 %
本次测试

In [13]:
# 将label值改为二分类以便画P-R图
label_111 = cp.deepcopy(label_000) 
label_222 = cp.deepcopy(label_000) 
label_333 = cp.deepcopy(label_000)

for i in range(len(label_000)):
    if label_000[i] != 0:
        label_000[i] = 4   
    else:
        label_000[i] = 0 
    if label_111[i] != 1:
        label_111[i] = 4 
    else:
        label_111[i] = 1 
    if label_222[i] != 2:
        label_222[i] = 4 
    else:
        label_222[i] = 2 
    if label_333[i] != 3:
        label_333[i] = 4 
    else:
        label_333[i] = 3

y_true0 = np.array(label_000)
y_scores = np.array(output_000[:,0])
precision000, recall000, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr000, tpr000, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_111)
y_scores = np.array(output_000[:,1])
precision111, recall111, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr111, tpr111, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_222)
y_scores = np.array(output_000[:,2])
precision222, recall222, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr222, tpr222, thersholds = roc_curve(y_true2, y_scores, pos_label=2)

y_true3 = np.array(label_333)
y_scores = np.array(output_000[:,3])
precision333, recall333, thresholds = precision_recall_curve(y_true3, y_scores,pos_label=3)
fpr333, tpr333, thersholds = roc_curve(y_true3, y_scores, pos_label=3)

pr_auc000,pr_auc111,pr_auc222,pr_auc333 = auc(recall000,precision000),auc(recall111,precision111),auc(recall222,precision222),auc(recall333,precision333)
roc_auc000,roc_auc111,roc_auc222,roc_auc333 = auc(fpr000,tpr000),auc(fpr111,tpr111),auc(fpr222,tpr222),auc(fpr333,tpr333)


In [45]:
#resqnet1 
Accbox1111 = []
Totallossbox1111 = []
Accbox2222 = []
Totallossbox2222 = []
rec1111_sam_train = [[],[],[],[]]
pre1111_sam_train = [[],[],[],[]]
rec1111_sam_test = [[],[],[],[]]
pre1111_sam_test = [[],[],[],[]]
f1111 = [[],[]]
f2222 = [[],[]]
f3333 = [[],[]]
f4444 = [[],[]]
weight1111 = [[],[],[]]
weight1111_gra = [[],[],[]]
 
#Net2 for mnist and fashionmnist
#mnist
class Net4(torch.nn.Module):
    def __init__(self):
        super(Net4, self).__init__()
        self.cbam1 = cbam(in_channel=8)
        self.cbam2 = cbam(in_channel=16)
        self.model1 = nn.Sequential(
        nn.Conv2d(in_channels = 3,out_channels = 8,kernel_size = 3,stride = 1,padding = 'same',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(8,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
       # nn.LeakyReLU(negative_slope = 0.5),
        )
            
        self.model2 = nn.Sequential(
        nn.Conv2d(in_channels = 32,out_channels = 4,kernel_size = 3,stride = 2,padding = 'valid',bias = False),
        nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(4,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
       # nn.LeakyReLU(negative_slope = 0.5),
            
        nn.Flatten(),
        )
        
        self.model3 = nn.Sequential(
        nn.Conv2d(in_channels = 8,out_channels = 16,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(16,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.model4 = nn.Sequential(
        nn.Conv2d(in_channels = 16,out_channels = 32,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        #nn.MaxPool2d(kernel_size = 3),
        nn.BatchNorm2d(32,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        
        self.model5 = nn.Sequential(
        nn.Conv2d(in_channels = 8,out_channels = 32,kernel_size = 1,stride = 2,padding = 'valid',bias = False),
        nn.MaxPool2d(kernel_size = 2),
        nn.BatchNorm2d(32,eps = 1e-5, momentum = 0.5,affine = True, track_running_stats = True),
        nn.LeakyReLU(negative_slope = 0.5),
        )
        self.fc1 = nn.Linear(4,4)
        self.fc2 = nn.Linear(12,12)
        self.fc = nn.Linear(4,4)

        for m in self.modules():
            if isinstance(m,nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm2d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.BatchNorm1d):
                nn.init.uniform_(m.weight,a = -1,b = 1)
      #          nn.init.constant_(m.bias, 0)
            elif isinstance(m,nn.Linear):
                nn.init.kaiming_uniform_(m.weight,a = 0.5, mode = 'fan_in', nonlinearity = 'leaky_relu')
          #      nn.init.constant_(m.bias, 0)
        
    def forward(self, x):
        x = self.model1(x)
        x = rotation(x,A=1,a=0.8,psi=math.pi/4)
        x1 = self.model5(x)
        x1 = rotation(x1,A=1,a=0.8,psi=math.pi/4)
       # print(x.shape)
        x = self.cbam1(x)+self.model3(x)
       # print(x.shape)
        x = self.cbam2(x)+self.model4(x)+x1
       # print(x.shape)
        x = self.model2(x)
        x = x.to(torch.float32)
        #x = rotation(self.fc1(x),A=1,a=0.8,psi=math.pi/4)
      #  noise = torch.Tensor(5).uniform_(-0.01,0.1)
      #  self.fc2.weight.data[0][0] = self.fc2.weight.data[0][0]+noise[0]
      #  self.fc2.weight.data[0][1] = self.fc2.weight.data[0][1]+noise[1]
      #  self.fc2.weight.data[0][2] = self.fc2.weight.data[0][2]+noise[2]
      #  self.fc2.weight.data[0][3] = self.fc2.weight.data[0][3]+noise[3]
      #  self.fc2.weight.data[0][4] = self.fc2.weight.data[0][4]+noise[4]
        #x = rotation(self.fc2(x),A=1.2,a=1,psi=math.pi/4)
        #x = rotation(self.fc2(x),A=1.8,a=1.6,psi=math.pi/4)
        #x = rotation(self.fc2(x),A=2,a=1.4,psi=math.pi/4)
        #x = rotation(self.fc2(x),A=1.4,a=1.4,psi=math.pi/4)
        x = gate_operation_res1(x,8,a,k)
        x = F.leaky_relu(self.fc1(x))
        output = self.fc(x)
        return output

model = Net4()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
summary(model,input_size =(3,32,32), batch_size = 32)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.005, weight_decay = 0.005)

#打印模型及模型优化时的参数
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

#保存模型参数
#state_mnist = {'net':model.state_dict(), 'optimizer':optimizer.state_dict(), 'epoch':epoch}
#torch.save(state_mnist,'C:/Users/Administrator/Desktop/CNN-mnist.pth')

#def add_noise(inputs,noise_factor):
#    noisy = inputs+torch.tensor(np.random.normal(0,1, size=( 3,32,32)), dtype=torch.float32)
#    return noisy 

def train(epoch,train_loader,trainset_num):
    perfect = 0
    Total = 0
    totalloss1 = 0.0
    eps = 1e-8
    global Accbox1111
    global Totallossbox1111
    TP_sam1_train, FP_sam1_train, FN_sam1_train = 0, 0, 0
    TP_sam2_train, FP_sam2_train, FN_sam2_train = 0, 0, 0
    TP_sam3_train, FP_sam3_train, FN_sam3_train = 0, 0, 0
    TP_sam4_train, FP_sam4_train, FN_sam4_train = 0, 0, 0
    global rec1111_sam_train 
    global pre1111_sam_train
    global f1111
    global f2222
    global f3333
    global f4444
    old_time_train = time.time()

    for batch_idx, data in enumerate(train_loader, 0):
        #print(batch_idx) #每一轮iteration，batch_idx从0到1874
        inputs, target = data
        #noisedata = add_noise(inputs,0.5)
        #print(inputs.shape)
        #print(target.shape)
        #print()
        #print(data)
        inputs, target = inputs.to(device),target.to(device)  #inputs为batchsize*1*28*28的向量，target为batchsize*1的向量，代表数字几
        #noisedata = noisedata.to(device)
        #print(inputs)
        #print(target)
        #print()
        outputs = model(inputs)
        target = target.type(torch.LongTensor)
        loss = criterion(outputs, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print(outputs.shape)
        #print(outputs)
        predict = torch.argmax(outputs.data, dim=1)
        #predict = predict.type(torch.FloatTensor)
        #target = target.type(torch.FloatTensor)
        #print(predict)
        #print(predict.shape)
        #print(target.shape)
        #target.requires_grad_(True)
        #predict.requires_grad_(True)
        Total += target.size(0) #每个iteration中labels.size(0)为batchsize数
        perfect += (predict == target).sum().item()
        totalloss1 += loss.item()
    print("total loss in training set")
    print(totalloss1/len(train_loader))
    Totallossbox1111 = np.append(Totallossbox1111,totalloss1/len(train_loader))
            
    print('Accuracy on training set: %.3f %%' % (100 * perfect / Total))
    Accbox1111 = np.append(Accbox1111, perfect / Total)
    current_time_train = time.time()
    print("本次训练集运行时间为" + str(current_time_train - old_time_train) + "s")
    for i in range(len(target)):
        if predict[i] == 0 and target[i] == 0:
            TP_sam1_train += 1
        if predict[i] == 0 and target[i] != 0:
            FP_sam1_train += 1
        if predict[i] != 0 and target[i] == 0:
            FN_sam1_train += 1

        if predict[i] == 1 and target[i] == 1:
            TP_sam2_train += 1
        if predict[i] == 1 and target[i] != 1:
            FP_sam2_train += 1
        if predict[i] != 1 and target[i] == 1:
            FN_sam2_train += 1    

        if predict[i] == 2 and target[i] == 2:
            TP_sam3_train += 1
        if predict[i] == 2 and target[i] != 2:
            FP_sam3_train += 1
        if predict[i] != 2 and target[i] == 2:
            FN_sam3_train += 1

        if predict[i] == 3 and target[i] == 3:
            TP_sam4_train += 1
        if predict[i] == 3 and target[i] != 3:
            FP_sam4_train += 1
        if predict[i] != 3 and target[i] == 3:
            FN_sam4_train += 1
            
    #计算四种样本的召回率与精确度
    P1 = TP_sam1_train/(TP_sam1_train+FP_sam1_train+eps)
    R1 = TP_sam1_train/(TP_sam1_train+FN_sam1_train+eps)
    F1_1train = 2*P1*R1/(P1+R1+eps)
    rec1111_sam_train[0] = np.append(rec1111_sam_train[0],P1)
    pre1111_sam_train[0] = np.append(pre1111_sam_train[0],R1)
    f1111[0] = np.append(f1111[0],F1_1train)
    
    P2 = TP_sam2_train/(TP_sam2_train+FP_sam2_train+eps)
    R2 = TP_sam2_train/(TP_sam2_train+FN_sam2_train+eps)
    F1_2train = 2*P2*R2/(P2+R2+eps)
    rec1111_sam_train[1] = np.append(rec1111_sam_train[1],P2)
    pre1111_sam_train[1] = np.append(pre1111_sam_train[1],R2)
    f2222[0] = np.append(f2222[0],F1_2train)
    
    P3 = TP_sam3_train/(TP_sam3_train+FP_sam3_train+eps)
    R3 = TP_sam3_train/(TP_sam3_train+FN_sam3_train+eps)
    F1_3train = 2*P3*R3/(P3+R3+eps)
    rec1111_sam_train[2] = np.append(rec1111_sam_train[2],P3)
    pre1111_sam_train[2] = np.append(pre1111_sam_train[2],R3)
    f3333[0] = np.append(f3333[0],F1_3train)
    
    P4 = TP_sam4_train/(TP_sam4_train+FP_sam4_train+eps)
    R4 = TP_sam4_train/(TP_sam4_train+FN_sam4_train+eps)
    F1_4train = 2*P4*R4/(P4+R4+eps)
    rec1111_sam_train[3] = np.append(rec1111_sam_train[3],P4)
    pre1111_sam_train[3] = np.append(pre1111_sam_train[3],R4)
    f4444[0] = np.append(f4444[0],F1_4train)

    
def test(test_loader,testset_num):
    correct = 0
    total = 0
    totalloss2 = 0.0
    eps = 1e-8
    lab = []
    out = []
    global Accbox2222
    global Totallossbox2222
    TP_sam1_test, FP_sam1_test, FN_sam1_test = 0, 0, 0
    TP_sam2_test, FP_sam2_test, FN_sam2_test = 0, 0, 0
    TP_sam3_test, FP_sam3_test, FN_sam3_test = 0, 0, 0
    TP_sam4_test, FP_sam4_test, FN_sam4_test = 0, 0, 0
    global rec1111_sam_test 
    global pre1111_sam_test
    global f1111
    global f2222
    global f3333
    global f4444
    
    with torch.no_grad():  # 测试不需要计算梯度
        old_time_test = time.time()
        for data in test_loader:
            images, labels = data
            #noisedata = add_noise(images,0.2)
            #print(images.shape) #batchsize*1*28*28
            #print(labels.shape) #batchsize*1
            images, labels = images.to(device),labels.to(device)
            #noisedata = noisedata.to(device)
            outputs = model(images)
            labels = labels.type(torch.LongTensor)
            loss = criterion(outputs,labels)
            lab = np.append(lab,labels.numpy())
            out = np.append(out,F.softmax(outputs).numpy())
            predicted = torch.argmax(outputs.data, dim=1)
            #此函数输出两个tensor,第一个tensor是每行的最大概率,第二个tensor是每行最大概率的索引,由于我们不需要获取最大概率的值，只要知道最大概率的是哪个类别即可,因此，我们只需要获取第二个tensor
            #print(predicted) #batchsize*1，均是0-9的数字
            #predicted = predicted.type(torch.FloatTensor)
            #labels = labels.type(torch.FloatTensor)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            #print(correct)
            totalloss2 += loss.item()
        Totallossbox2222 = np.append(Totallossbox2222,totalloss2/len(test_loader))
        print("total loss in testing set")
        print(totalloss2/len(test_loader))
        #totalloss2 = 0.0       
        print('Accuracy on test set: %.3f %%' % (100 * correct / total))
        Accbox2222 = np.append(Accbox2222, correct / total)
        current_time_test = time.time()
        print("本次测试集运行时间为" + str(current_time_test - old_time_test) + "s")
        print()
        print()
        print()
        for i in range(len(labels)):
            if predicted[i] == 0 and labels[i] == 0:
                TP_sam1_test += 1
            if predicted[i] == 0 and labels[i] != 0:
                FP_sam1_test += 1
            if predicted[i] != 0 and labels[i] == 0:
                FN_sam1_test += 1

            if predicted[i] == 1 and labels[i] == 1:
                TP_sam2_test += 1
            if predicted[i] == 1 and labels[i] != 1:
                FP_sam2_test += 1
            if predicted[i] != 1 and labels[i] == 1:
                FN_sam2_test += 1    

            if predicted[i] == 2 and labels[i] == 2:
                TP_sam3_test += 1
            if predicted[i] == 2 and labels[i] != 2:
                FP_sam3_test += 1
            if predicted[i] != 2 and labels[i] == 2:
                FN_sam3_test += 1

            if predicted[i] == 3 and labels[i] == 3:
                TP_sam4_test += 1
            if predicted[i] == 3 and labels[i] != 3:
                FP_sam4_test += 1
            if predicted[i] != 3 and labels[i] == 3:
                FN_sam4_test += 1
    
    P1 = TP_sam1_test/(TP_sam1_test+FP_sam1_test+eps)
    R1 = TP_sam1_test/(TP_sam1_test+FN_sam1_test+eps)
    F1_1test = 2*P1*R1/(P1+R1+eps)
    rec1111_sam_test[0] = np.append(rec1111_sam_test[0],P1)
    pre1111_sam_test[0] = np.append(pre1111_sam_test[0],R1)
    f1111[1] = np.append(f1111[1],F1_1test)
    
    P2 = TP_sam2_test/(TP_sam2_test+FP_sam2_test+eps)
    R2 = TP_sam2_test/(TP_sam2_test+FN_sam2_test+eps)
    F1_2test = 2*P2*R2/(P2+R2+eps)
    rec1111_sam_test[1] = np.append(rec1111_sam_test[1],P2)
    pre1111_sam_test[1] = np.append(pre1111_sam_test[1],R2)
    f2222[1] = np.append(f2222[1],F1_2test)
    
    P3 = TP_sam3_test/(TP_sam3_test+FP_sam3_test+eps)
    R3 = TP_sam3_test/(TP_sam3_test+FN_sam3_test+eps)
    F1_3test = 2*P3*R3/(P3+R3+eps)
    rec1111_sam_test[2] = np.append(rec1111_sam_test[2],P3)
    pre1111_sam_test[2] = np.append(pre1111_sam_test[2],R3)
    f3333[1] = np.append(f3333[1],F1_3test)

    P4 = TP_sam4_test/(TP_sam4_test+FP_sam4_test+eps)
    R4 = TP_sam4_test/(TP_sam4_test+FN_sam4_test+eps)
    F1_4test = 2*P4*R4/(P4+R4+eps)
    rec1111_sam_test[3] = np.append(rec1111_sam_test[3],P4)
    pre1111_sam_test[3] = np.append(pre1111_sam_test[3],R4)
    f4444[1] = np.append(f4444[1],F1_4test)
    return lab,out   


if __name__ == '__main__':
    for i in range(epoch):
        print('epoch:',i+1)
        #b = torch.tensor(np.random.normal(0.01,0.01, size=(5)), dtype=torch.float32)
        train(epoch,train_loader,2000)
        weight1111[0] = np.append(weight1111[0],model.fc1.weight[0][0].item())
        weight1111[1] = np.append(weight1111[1],model.fc1.weight[0][1].item())
        weight1111[2] = np.append(weight1111[2],model.fc1.weight[0][2].item())
        if i != 0:
            weight1111_gra[0] = np.append(weight1111_gra[0],model.fc1.weight.grad[0][0].item())
            weight1111_gra[1] = np.append(weight1111_gra[1],model.fc1.weight.grad[0][1].item())
            weight1111_gra[2] = np.append(weight1111_gra[2],model.fc1.weight.grad[0][2].item())
        label_0000,output_0000 = test(test_loader,400)
    output_0000 = output_0000.reshape(-1,4)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [32, 8, 32, 32]             216
       BatchNorm2d-2            [32, 8, 32, 32]              16
            Conv2d-3           [32, 32, 16, 16]             256
         MaxPool2d-4             [32, 32, 8, 8]               0
       BatchNorm2d-5             [32, 32, 8, 8]              64
         LeakyReLU-6             [32, 32, 8, 8]               0
 AdaptiveMaxPool2d-7              [32, 8, 1, 1]               0
 AdaptiveAvgPool2d-8              [32, 8, 1, 1]               0
            Linear-9                    [32, 2]              16
           Linear-10                    [32, 2]              16
        LeakyReLU-11                    [32, 2]               0
        LeakyReLU-12                    [32, 2]               0
           Linear-13                    [32, 8]              16
           Linear-14                   

C:\Users\15850\AppData\Local\Temp\ipykernel_3296\3644786907.py:282: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  out = np.append(out,F.softmax(outputs).numpy())


total loss in testing set
1.1030772924423218
Accuracy on test set: 52.000 %
本次测试集运行时间为0.4960002899169922s



epoch: 2
total loss in training set
1.0631070401933458
Accuracy on training set: 52.200 %
本次训练集运行时间为5.754203796386719s
total loss in testing set
0.91822749376297
Accuracy on test set: 62.500 %
本次测试集运行时间为0.5080022811889648s



epoch: 3
total loss in training set
0.9262665292573353
Accuracy on training set: 62.000 %
本次训练集运行时间为5.752168893814087s
total loss in testing set
0.8235034621678866
Accuracy on test set: 68.000 %
本次测试集运行时间为0.5219976902008057s



epoch: 4
total loss in training set
0.8477989132442172
Accuracy on training set: 65.600 %
本次训练集运行时间为5.844253778457642s
total loss in testing set
0.7849499674943777
Accuracy on test set: 70.000 %
本次测试集运行时间为0.5596117973327637s



epoch: 5
total loss in training set
0.7764092300619397
Accuracy on training set: 69.200 %
本次训练集运行时间为5.847265720367432s
total loss in testing set
0.7195505591539236
Accuracy on test set: 73.500 %
本次测试集运行时间为0.53

total loss in testing set
0.687931450513693
Accuracy on test set: 77.500 %
本次测试集运行时间为0.5310029983520508s



epoch: 38
total loss in training set
0.4251788490348392
Accuracy on training set: 83.400 %
本次训练集运行时间为5.788639307022095s
total loss in testing set
0.658816404067553
Accuracy on test set: 75.750 %
本次测试集运行时间为0.5490922927856445s



epoch: 39
total loss in training set
0.43606276337116484
Accuracy on training set: 84.050 %
本次训练集运行时间为5.736276149749756s
total loss in testing set
0.6343439656954545
Accuracy on test set: 77.500 %
本次测试集运行时间为0.5289981365203857s



epoch: 40
total loss in training set
0.42393206769511815
Accuracy on training set: 84.000 %
本次训练集运行时间为5.762322187423706s
total loss in testing set
0.6107244491577148
Accuracy on test set: 79.750 %
本次测试集运行时间为0.5267715454101562s



epoch: 41
total loss in training set
0.3746894061092346
Accuracy on training set: 86.100 %
本次训练集运行时间为5.800108194351196s
total loss in testing set
0.6371039576255358
Accuracy on test set: 77.000 %
本次测试集运行时

total loss in testing set
0.7233510544666877
Accuracy on test set: 75.250 %
本次测试集运行时间为0.5389792919158936s



epoch: 74
total loss in training set
0.3128482410832057
Accuracy on training set: 88.150 %
本次训练集运行时间为5.899987697601318s
total loss in testing set
0.691124861056988
Accuracy on test set: 76.750 %
本次测试集运行时间为0.5316829681396484s



epoch: 75
total loss in training set
0.3392738997936249
Accuracy on training set: 87.850 %
本次训练集运行时间为5.828403949737549s
total loss in testing set
0.6461961888349973
Accuracy on test set: 78.000 %
本次测试集运行时间为0.5750038623809814s



epoch: 76
total loss in training set
0.3174845592843162
Accuracy on training set: 88.250 %
本次训练集运行时间为5.757100582122803s
total loss in testing set
0.6623742534564092
Accuracy on test set: 77.000 %
本次测试集运行时间为0.5670039653778076s



epoch: 77
total loss in training set
0.33042542293431265
Accuracy on training set: 87.900 %
本次训练集运行时间为5.748922348022461s
total loss in testing set
0.6778821142820212
Accuracy on test set: 76.000 %
本次测试集运行时

...

total loss in testing set
0.6172739198574653
Accuracy on test set: 79.250 %
本次测试集运行时间为0.5879409313201904s



epoch: 146
total loss in training set
0.2728419651587804
Accuracy on training set: 90.100 %
本次训练集运行时间为5.81837797164917s
total loss in testing set
0.6745560421393468
Accuracy on test set: 77.000 %
本次测试集运行时间为0.6730010509490967s



epoch: 147
total loss in training set
0.2840804403263425
Accuracy on training set: 89.100 %
本次训练集运行时间为5.84466290473938s
total loss in testing set
0.59517714037345
Accuracy on test set: 79.250 %
本次测试集运行时间为0.5399959087371826s



epoch: 148
total loss in training set
0.2763513832811325
Accuracy on training set: 90.150 %
本次训练集运行时间为5.706976413726807s
total loss in testing set
0.660194449699842
Accuracy on test set: 76.500 %
本次测试集运行时间为0.5540058612823486s



epoch: 149
total loss in training set
0.2598084614626945
Accuracy on training set: 90.050 %
本次训练集运行时间为5.853653192520142s
total loss in testing set
0.6106032637449411
Accuracy on test set: 79.250 %
本次测试集运行时间

total loss in testing set
0.6831969355161374
Accuracy on test set: 79.000 %
本次测试集运行时间为0.5530054569244385s



epoch: 182
total loss in training set
0.2529647285266528
Accuracy on training set: 90.300 %
本次训练集运行时间为5.765878438949585s
total loss in testing set
0.7688557803630829
Accuracy on test set: 74.750 %
本次测试集运行时间为0.5180048942565918s



epoch: 183
total loss in training set
0.2497064359486103
Accuracy on training set: 90.650 %
本次训练集运行时间为5.7586119174957275s
total loss in testing set
0.6657336491804856
Accuracy on test set: 78.750 %
本次测试集运行时间为0.543940544128418s



epoch: 184
total loss in training set
0.26428927008121733
Accuracy on training set: 90.800 %
本次训练集运行时间为5.803321838378906s
total loss in testing set
0.7007459654257848
Accuracy on test set: 76.250 %
本次测试集运行时间为0.5489943027496338s



epoch: 185
total loss in training set
0.23619597009013568
Accuracy on training set: 91.050 %
本次训练集运行时间为5.733391284942627s
total loss in testing set
0.7101173148705409
Accuracy on test set: 76.250 %
本次

total loss in testing set
0.7068561200912182
Accuracy on test set: 77.500 %
本次测试集运行时间为0.5959899425506592s



epoch: 218
total loss in training set
0.2737704934108825
Accuracy on training set: 90.100 %
本次训练集运行时间为5.887240648269653s
total loss in testing set
0.686199392263706
Accuracy on test set: 76.500 %
本次测试集运行时间为0.5370032787322998s



epoch: 219
total loss in training set
0.25823370639293913
Accuracy on training set: 90.000 %
本次训练集运行时间为5.7319865226745605s
total loss in testing set
0.7100226581096649
Accuracy on test set: 77.250 %
本次测试集运行时间为0.5413084030151367s



epoch: 220
total loss in training set
0.28625589952109354
Accuracy on training set: 88.850 %
本次训练集运行时间为5.677766799926758s
total loss in testing set
0.6912584465283614
Accuracy on test set: 75.750 %
本次测试集运行时间为0.49900293350219727s



epoch: 221
total loss in training set
0.2655430768453886
Accuracy on training set: 90.150 %
本次训练集运行时间为5.705039024353027s
total loss in testing set
0.8119587646080897
Accuracy on test set: 75.000 %
本

total loss in testing set
0.7731097730306479
Accuracy on test set: 74.750 %
本次测试集运行时间为0.5420017242431641s



epoch: 254
total loss in training set
0.2511910195388491
Accuracy on training set: 90.450 %
本次训练集运行时间为5.79319167137146s
total loss in testing set
0.6725995884491847
Accuracy on test set: 77.500 %
本次测试集运行时间为0.5350015163421631s



epoch: 255
total loss in training set
0.22014447597284165
Accuracy on training set: 92.050 %
本次训练集运行时间为5.790569305419922s
total loss in testing set
0.7826901353322543
Accuracy on test set: 75.000 %
本次测试集运行时间为0.5840020179748535s



epoch: 256
total loss in training set
0.24346013343523418
Accuracy on training set: 91.200 %
本次训练集运行时间为5.826364755630493s
total loss in testing set
0.7290653196664957
Accuracy on test set: 75.500 %
本次测试集运行时间为0.5411558151245117s



epoch: 257
total loss in training set
0.2862019453729902
Accuracy on training set: 89.800 %
本次训练集运行时间为5.818495512008667s
total loss in testing set
0.6695802097137158
Accuracy on test set: 79.000 %
本次测

total loss in testing set
0.7068021114055927
Accuracy on test set: 76.500 %
本次测试集运行时间为0.5750002861022949s



epoch: 290
total loss in training set
0.1792549882971105
Accuracy on training set: 93.350 %
本次训练集运行时间为5.731110334396362s
total loss in testing set
0.737686350941658
Accuracy on test set: 77.250 %
本次测试集运行时间为0.5599954128265381s



epoch: 291
total loss in training set
0.2629645889595387
Accuracy on training set: 90.150 %
本次训练集运行时间为5.892203330993652s
total loss in testing set
0.8171346600239093
Accuracy on test set: 75.750 %
本次测试集运行时间为0.5529999732971191s



epoch: 292
total loss in training set
0.24560757720517734
Accuracy on training set: 90.750 %
本次训练集运行时间为6.390021800994873s
total loss in testing set
0.7665085930090684
Accuracy on test set: 75.250 %
本次测试集运行时间为0.5000033378601074s



epoch: 293
total loss in training set
0.2085988658761221
Accuracy on training set: 92.050 %
本次训练集运行时间为5.793869495391846s
total loss in testing set
0.7960957197042612
Accuracy on test set: 76.500 %
本次测试

In [15]:
label_1111 = cp.deepcopy(label_0000) 
label_2222 = cp.deepcopy(label_0000) 
label_3333 = cp.deepcopy(label_0000)

for i in range(len(label_0000)):
    if label_0000[i] != 0:
        label_0000[i] = 4   
    else:
        label_0000[i] = 0 
    if label_1111[i] != 1:
        label_1111[i] = 4 
    else:
        label_1111[i] = 1 
    if label_2222[i] != 2:
        label_2222[i] = 4 
    else:
        label_2222[i] = 2 
    if label_3333[i] != 3:
        label_3333[i] = 4 
    else:
        label_3333[i] = 3

y_true0 = np.array(label_0000)
y_scores = np.array(output_0000[:,0])
precision0000, recall0000, thresholds = precision_recall_curve(y_true0, y_scores,pos_label=0)
fpr0000, tpr0000, thersholds = roc_curve(y_true0, y_scores, pos_label=0)

y_true1 = np.array(label_1111)
y_scores = np.array(output_0000[:,1])
precision1111, recall1111, thresholds = precision_recall_curve(y_true1, y_scores,pos_label=1)
fpr1111, tpr1111, thersholds = roc_curve(y_true1, y_scores, pos_label=1)

y_true2 = np.array(label_2222)
y_scores = np.array(output_0000[:,2])
precision2222, recall2222, thresholds = precision_recall_curve(y_true2, y_scores,pos_label=2)
fpr2222, tpr2222, thersholds = roc_curve(y_true2, y_scores, pos_label=2)

y_true3 = np.array(label_3333)
y_scores = np.array(output_0000[:,3])
precision3333, recall3333, thresholds = precision_recall_curve(y_true3, y_scores,pos_label=3)
fpr3333, tpr3333, thersholds = roc_curve(y_true3, y_scores, pos_label=3)

pr_auc0000,pr_auc1111,pr_auc2222,pr_auc3333 = auc(recall0000,precision0000),auc(recall1111,precision1111),auc(recall2222,precision2222),auc(recall3333,precision3333)
roc_auc0000,roc_auc1111,roc_auc2222,roc_auc3333 = auc(fpr0000,tpr0000),auc(fpr1111,tpr1111),auc(fpr2222,tpr2222),auc(fpr3333,tpr3333)


In [16]:
pr_aucQRCNN = (pr_auc0+pr_auc1+pr_auc2+pr_auc3)/4
pr_aucCNN1 = (pr_auc00+pr_auc11+pr_auc22+pr_auc33)/4
pr_aucQDCNN = (pr_auc000+pr_auc111+pr_auc222+pr_auc333)/4
pr_aucCNN2 = (pr_auc0000+pr_auc1111+pr_auc2222+pr_auc3333)/4

roc_aucQRCNN = (roc_auc0+roc_auc1+roc_auc2+roc_auc3)/4
roc_aucCNN1 = (roc_auc00+roc_auc11+roc_auc22+roc_auc33)/4
roc_aucQDCNN = (roc_auc000+roc_auc111+roc_auc222+roc_auc333)/4
roc_aucCNN2 = (roc_auc0000+roc_auc1111+roc_auc2222+roc_auc3333)/4

print('4种神经网络P-R曲线面积为:',pr_aucQRCNN,pr_aucCNN1,pr_aucQDCNN,pr_aucCNN2)
print('4种神经网络ROC曲线面积为:',roc_aucQRCNN,roc_aucCNN1,roc_aucQDCNN,roc_aucCNN2)

%matplotlib qt5
plt.subplot(221)
plt.plot(recall0,precision0,label = "P-R curve of QRCNN",lw = 3)
plt.plot(recall00,precision00,label = "P-R curve of CNN(leaky-relu)",lw = 3)
plt.plot(recall000,precision000,label = "P-R curve of QDCNN",lw = 3)
plt.plot(recall0000,precision0000,label = "P-R curve of CNN(rot-relu)",lw = 3)
plt.title("precision-recall plot of CIAFR100 of sample 0")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='lower right')
plt.grid()

plt.subplot(222)
plt.plot(recall1,precision1,label = "P-R curve of QRCNN",lw = 3)
plt.plot(recall11,precision11,label = "P-R curve of CNN(leaky-relu)",lw = 3)
plt.plot(recall111,precision111,label = "P-R curve of QDCNN",lw = 3)
plt.plot(recall1111,precision1111,label = "P-R curve of CNN(rot-relu)",lw = 3)
plt.title("precision-recall plot of CIFAR100 of sample 1")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='lower right')
plt.grid()

plt.subplot(223)
plt.plot(recall2,precision2,label = "P-R curve of QRCNN",lw = 3)
plt.plot(recall22,precision22,label = "P-R curve of CNN(leaky-relu)",lw = 3)
plt.plot(recall222,precision222,label = "P-R curve of QDCNN",lw = 3)
plt.plot(recall2222,precision2222,label = "P-R curve of CNN(rot-relu)",lw = 3)
plt.title("precision-recall plot of CIFAR100 of sample 2")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='lower right')
plt.grid()

plt.subplot(224)
plt.plot(recall3,precision3,label = "P-R curve of QRCNN",lw = 3)
plt.plot(recall33,precision33,label = "P-R curve of CNN(leaky-relu)",lw = 3)
plt.plot(recall333,precision333,label = "P-R curve of QDCNN",lw = 3)
plt.plot(recall3333,precision3333,label = "P-R curve of CNN(rot-relu)",lw = 3)
plt.title("precision-recall plot of CIFAR100 of sample 0")
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='lower right')
plt.grid() 

4种神经网络P-R曲线面积为: 0.8683554960334037 0.8488897451412518 0.8412558128145122 0.8620266845242225
4种神经网络ROC曲线面积为: 0.9470000000000001 0.9332083333333334 0.9392416666666668 0.9433333333333332


In [17]:
%matplotlib qt5
plt.subplot(221)
plt.plot(fpr0,tpr0,label = "ROC curve of QRCNN",lw = 3)
plt.plot(fpr00,tpr00,label = "ROC curve of CNN(leaky-relu)",lw = 3)
plt.plot(fpr000,tpr000,label = "ROC curve of QDCNN",lw = 3)
plt.plot(fpr0000,tpr0000,label = "ROC curve of CNN(rot-relu)",lw = 3)
plt.title("ROC plot of CIFAR100 of sample 0")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend()
plt.grid()

plt.subplot(222)
plt.plot(fpr1,tpr1,label = "ROC curve of QRCNN",lw = 3)
plt.plot(fpr11,tpr11,label = "ROC curve of CNN(leaky-relu)",lw = 3)
plt.plot(fpr111,tpr111,label = "ROC curve of QDCNN",lw = 3)
plt.plot(fpr1111,tpr1111,label = "ROC curve of CNN(rot-relu)",lw = 3)
plt.title("ROC plot of CIFAR100 of sample 1")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend()
plt.grid()

plt.subplot(223)
plt.plot(fpr2,tpr2,label = "ROC curve of QRCNN",lw = 3)
plt.plot(fpr22,tpr22,label = "ROC curve of CNN(leaky-relu)",lw = 3)
plt.plot(fpr222,tpr222,label = "ROC curve of QDCNN",lw = 3)
plt.plot(fpr2222,tpr2222,label = "ROC curve of CNN(rot-relu)",lw = 3)
plt.title("ROC plot of CIFAR100 of sample 2")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend()
plt.grid()

plt.subplot(224)
plt.plot(fpr3,tpr3,label = "ROC curve of QRCNN",lw = 3)
plt.plot(fpr33,tpr33,label = "ROC curve of CNN(leaky-relu)",lw = 3)
plt.plot(fpr333,tpr333,label = "ROC curve of QDCNN",lw = 3)
plt.plot(fpr3333,tpr3333,label = "ROC curve of CNN(rot-relu)",lw = 3)
plt.title("ROC plot of CIFAR100 of sample 3")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='lower left')
plt.grid()

In [18]:
x1 = np.linspace(1,epoch,epoch)
x2 = np.linspace(2,epoch,epoch-1)
xs = np.linspace(1,epoch,1000000)
xss = np.linspace(2,epoch,1000000)
plt.figure(figsize = (18,6))

#rec召回率，pre精确度，前面的数字代表第几类样本
fitloss1 = make_interp_spline(x1,Totallossbox1)
fitloss2 = make_interp_spline(x1,Totallossbox2)
fitacc1 = make_interp_spline(x1,Accbox1)
fitacc2 = make_interp_spline(x1,Accbox2)
fitrec1_sam1_train = make_interp_spline(x1,rec1_sam_train[0])
fitrec1_sam2_train = make_interp_spline(x1,rec1_sam_train[1])
fitrec1_sam3_train = make_interp_spline(x1,rec1_sam_train[2])
fitrec1_sam4_train = make_interp_spline(x1,rec1_sam_train[3])
fitpre1_sam1_train = make_interp_spline(x1,pre1_sam_train[0])
fitpre1_sam2_train = make_interp_spline(x1,pre1_sam_train[1])
fitpre1_sam3_train = make_interp_spline(x1,pre1_sam_train[2])
fitpre1_sam4_train = make_interp_spline(x1,pre1_sam_train[3])
fitrec1_sam1_test = make_interp_spline(x1,rec1_sam_test[0])
fitrec1_sam2_test = make_interp_spline(x1,rec1_sam_test[1])
fitrec1_sam3_test = make_interp_spline(x1,rec1_sam_test[2])
fitrec1_sam4_test = make_interp_spline(x1,rec1_sam_test[3])
fitpre1_sam1_test = make_interp_spline(x1,pre1_sam_test[0])
fitpre1_sam2_test = make_interp_spline(x1,pre1_sam_test[1])
fitpre1_sam3_test = make_interp_spline(x1,pre1_sam_test[2])
fitpre1_sam4_test = make_interp_spline(x1,pre1_sam_test[3])
#这里默认是F1-score,数字代表第几类样本
fit_f1_train = make_interp_spline(x1,f1[0])
fit_f1_test = make_interp_spline(x1,f1[1])
fit_f2_train = make_interp_spline(x1,f2[0])
fit_f2_test = make_interp_spline(x1,f2[1])
fit_f3_train = make_interp_spline(x1,f3[0])
fit_f3_test = make_interp_spline(x1,f3[1])
fit_f4_train = make_interp_spline(x1,f4[0])
fit_f4_test = make_interp_spline(x1,f4[1])
fit_weight1 = make_interp_spline(x1,weight1[0])
fit_weight2 = make_interp_spline(x1,weight1[1])
fit_weight3 = make_interp_spline(x1,weight1[2])
fit_weight_gra1 = make_interp_spline(x2,weight1_gra[0])
fit_weight_gra2 = make_interp_spline(x2,weight1_gra[1])
fit_weight_gra3 = make_interp_spline(x2,weight1_gra[2])
ys1 = fitloss1(xs)
ys2 = fitloss2(xs)
ys3 = fitacc1(xs)
ys4 = fitacc2(xs)
ys5 = fitrec1_sam1_train(xs)
ys6 = fitrec1_sam2_train(xs)
ys7 = fitrec1_sam3_train(xs)
ys8 = fitrec1_sam4_train(xs)
ys9 = fitpre1_sam1_train(xs)
ys10 = fitpre1_sam2_train(xs)
ys11 = fitpre1_sam3_train(xs)
ys12 = fitpre1_sam4_train(xs)
ys13 = fitrec1_sam1_test(xs)
ys14 = fitrec1_sam2_test(xs)
ys15 = fitrec1_sam3_test(xs)
ys16 = fitrec1_sam4_test(xs)
ys17 = fitpre1_sam1_test(xs)
ys18 = fitpre1_sam2_test(xs)
ys19 = fitpre1_sam3_test(xs)
ys20 = fitpre1_sam4_test(xs)
ys21 = fit_f1_train(xs)
ys22 = fit_f1_test(xs)
ys23 = fit_f2_train(xs)
ys24 = fit_f2_test(xs)
ys25 = fit_f3_train(xs)
ys26 = fit_f3_test(xs)
ys27 = fit_f4_train(xs)
ys28 = fit_f4_test(xs)
ys29 = fit_weight1(xs)
ys30 = fit_weight2(xs)
ys31 = fit_weight3(xs)
ys32 = fit_weight_gra1(xss)
ys33 = fit_weight_gra2(xss)
ys34 = fit_weight_gra3(xss)




fitloss11 = make_interp_spline(x1,Totallossbox11)
fitloss22 = make_interp_spline(x1,Totallossbox22)
fitacc11 = make_interp_spline(x1,Accbox11)
fitacc22 = make_interp_spline(x1,Accbox22)
fitrec11_sam1_train = make_interp_spline(x1,rec11_sam_train[0])
fitrec11_sam2_train = make_interp_spline(x1,rec11_sam_train[1])
fitrec11_sam3_train = make_interp_spline(x1,rec11_sam_train[2])
fitrec11_sam4_train = make_interp_spline(x1,rec11_sam_train[3])
fitpre11_sam1_train = make_interp_spline(x1,pre11_sam_train[0])
fitpre11_sam2_train = make_interp_spline(x1,pre11_sam_train[1])
fitpre11_sam3_train = make_interp_spline(x1,pre11_sam_train[2])
fitpre11_sam4_train = make_interp_spline(x1,pre11_sam_train[3])
fitrec11_sam1_test = make_interp_spline(x1,rec11_sam_test[0])
fitrec11_sam2_test = make_interp_spline(x1,rec11_sam_test[1])
fitrec11_sam3_test = make_interp_spline(x1,rec11_sam_test[2])
fitrec11_sam4_test = make_interp_spline(x1,rec11_sam_test[3])
fitpre11_sam1_test = make_interp_spline(x1,pre11_sam_test[0])
fitpre11_sam2_test = make_interp_spline(x1,pre11_sam_test[1])
fitpre11_sam3_test = make_interp_spline(x1,pre11_sam_test[2])
fitpre11_sam4_test = make_interp_spline(x1,pre11_sam_test[3])
fit_f11_train = make_interp_spline(x1,f11[0])
fit_f11_test = make_interp_spline(x1,f11[1])
fit_f22_train = make_interp_spline(x1,f22[0])
fit_f22_test = make_interp_spline(x1,f22[1])
fit_f33_train = make_interp_spline(x1,f33[0])
fit_f33_test = make_interp_spline(x1,f33[1])
fit_f44_train = make_interp_spline(x1,f44[0])
fit_f44_test = make_interp_spline(x1,f44[1])
fit_weight11 = make_interp_spline(x1,weight11[0])
fit_weight22 = make_interp_spline(x1,weight11[1])
fit_weight33 = make_interp_spline(x1,weight11[2])
fit_weight_gra11 = make_interp_spline(x2,weight11_gra[0])
fit_weight_gra22 = make_interp_spline(x2,weight11_gra[1])
fit_weight_gra33 = make_interp_spline(x2,weight11_gra[2])
yss11 = fitloss11(xs)
yss22 = fitloss22(xs)
ys33 = fitacc11(xs)
ys44 = fitacc22(xs)
ys55 = fitrec11_sam1_train(xs)
ys66 = fitrec11_sam2_train(xs)
ys77 = fitrec11_sam3_train(xs)
ys88 = fitrec11_sam4_train(xs)
ys99 = fitpre11_sam1_train(xs)
ys100 = fitpre11_sam2_train(xs)
yss111 = fitpre11_sam3_train(xs)
ys122 = fitpre11_sam4_train(xs)
ys133 = fitrec11_sam1_test(xs)
ys144 = fitrec11_sam2_test(xs)
ys155 = fitrec11_sam3_test(xs)
ys166 = fitrec11_sam4_test(xs)
ys177 = fitpre11_sam1_test(xs)
ys188 = fitpre11_sam2_test(xs)
ys199 = fitpre11_sam3_test(xs)
ys200 = fitpre11_sam4_test(xs)
ys211 = fit_f11_train(xs)
yss222 = fit_f11_test(xs)
ys233 = fit_f22_train(xs)
ys244 = fit_f22_test(xs)
ys255 = fit_f33_train(xs)
ys266 = fit_f33_test(xs)
ys277 = fit_f44_train(xs)
ys288 = fit_f44_test(xs)
ys299 = fit_weight11(xs)
ys300 = fit_weight22(xs)
ys311 = fit_weight33(xs)
ys322 = fit_weight_gra11(xss)
ys333 = fit_weight_gra22(xss)
ys344 = fit_weight_gra33(xss)




fitloss111 = make_interp_spline(x1,Totallossbox111)
fitloss222 = make_interp_spline(x1,Totallossbox222)
fitacc111 = make_interp_spline(x1,Accbox111)
fitacc222 = make_interp_spline(x1,Accbox222)
fitrec111_sam1_train = make_interp_spline(x1,rec111_sam_train[0])
fitrec111_sam2_train = make_interp_spline(x1,rec111_sam_train[1])
fitrec111_sam3_train = make_interp_spline(x1,rec111_sam_train[2])
fitrec111_sam4_train = make_interp_spline(x1,rec111_sam_train[3])
fitpre111_sam1_train = make_interp_spline(x1,pre111_sam_train[0])
fitpre111_sam2_train = make_interp_spline(x1,pre111_sam_train[1])
fitpre111_sam3_train = make_interp_spline(x1,pre111_sam_train[2])
fitpre111_sam4_train = make_interp_spline(x1,pre111_sam_train[3])
fitrec111_sam1_test = make_interp_spline(x1,rec111_sam_test[0])
fitrec111_sam2_test = make_interp_spline(x1,rec111_sam_test[1])
fitrec111_sam3_test = make_interp_spline(x1,rec111_sam_test[2])
fitrec111_sam4_test = make_interp_spline(x1,rec111_sam_test[3])
fitpre111_sam1_test = make_interp_spline(x1,pre111_sam_test[0])
fitpre111_sam2_test = make_interp_spline(x1,pre111_sam_test[1])
fitpre111_sam3_test = make_interp_spline(x1,pre111_sam_test[2])
fitpre111_sam4_test = make_interp_spline(x1,pre111_sam_test[3])
fit_f111_train = make_interp_spline(x1,f111[0])
fit_f111_test = make_interp_spline(x1,f111[1])
fit_f222_train = make_interp_spline(x1,f222[0])
fit_f222_test = make_interp_spline(x1,f222[1])
fit_f333_train = make_interp_spline(x1,f333[0])
fit_f333_test = make_interp_spline(x1,f333[1])
fit_f444_train = make_interp_spline(x1,f444[0])
fit_f444_test = make_interp_spline(x1,f444[1])
fit_weight111 = make_interp_spline(x1,weight111[0])
fit_weight222 = make_interp_spline(x1,weight111[1])
fit_weight333 = make_interp_spline(x1,weight111[2])
fit_weight_gra111 = make_interp_spline(x2,weight111_gra[0])
fit_weight_gra222 = make_interp_spline(x2,weight111_gra[1])
fit_weight_gra333 = make_interp_spline(x2,weight111_gra[2])
ys111 = fitloss111(xs)
ys222 = fitloss222(xs)
ys333 = fitacc111(xs)
ys444 = fitacc222(xs)
ys555 = fitrec111_sam1_train(xs)
ys666 = fitrec111_sam2_train(xs)
ys777 = fitrec111_sam3_train(xs)
ys888 = fitrec111_sam4_train(xs)
ys999 = fitpre111_sam1_train(xs)
ys1000 = fitpre111_sam2_train(xs)
ys1111 = fitpre111_sam3_train(xs)
ys1222 = fitpre111_sam4_train(xs)
ys1333 = fitrec111_sam1_test(xs)
ys1444 = fitrec111_sam2_test(xs)
ys1555 = fitrec111_sam3_test(xs)
ys1666 = fitrec111_sam4_test(xs)
ys1777 = fitpre111_sam1_test(xs)
ys1888 = fitpre111_sam2_test(xs)
ys1999 = fitpre111_sam3_test(xs)
ys2000 = fitpre111_sam4_test(xs)
ys2111 = fit_f111_train(xs)
ys2222 = fit_f111_test(xs)
ys2333 = fit_f222_train(xs)
ys2444 = fit_f222_test(xs)
ys2555 = fit_f333_train(xs)
ys2666 = fit_f333_test(xs)
ys2777 = fit_f444_train(xs)
ys2888 = fit_f444_test(xs)
ys2999 = fit_weight111(xs)
ys3000 = fit_weight222(xs)
ys3111 = fit_weight333(xs)
ys3222 = fit_weight_gra111(xss)
ys3333 = fit_weight_gra222(xss)
ys3444 = fit_weight_gra333(xss)




fitloss111e = make_interp_spline(x1,Totallossbox1111)
fitloss222e = make_interp_spline(x1,Totallossbox2222)
fitacc111e = make_interp_spline(x1,Accbox1111)
fitacc222e = make_interp_spline(x1,Accbox2222)
fitrec111e_sam1_train = make_interp_spline(x1,rec1111_sam_train[0])
fitrec111e_sam2_train = make_interp_spline(x1,rec1111_sam_train[1])
fitrec111e_sam3_train = make_interp_spline(x1,rec1111_sam_train[2])
fitrec111e_sam4_train = make_interp_spline(x1,rec1111_sam_train[3])
fitpre111e_sam1_train = make_interp_spline(x1,pre1111_sam_train[0])
fitpre111e_sam2_train = make_interp_spline(x1,pre1111_sam_train[1])
fitpre111e_sam3_train = make_interp_spline(x1,pre1111_sam_train[2])
fitpre111e_sam4_train = make_interp_spline(x1,pre1111_sam_train[3])
fitrec111e_sam1_test = make_interp_spline(x1,rec1111_sam_test[0])
fitrec111e_sam2_test = make_interp_spline(x1,rec1111_sam_test[1])
fitrec111e_sam3_test = make_interp_spline(x1,rec1111_sam_test[2])
fitrec111e_sam4_test = make_interp_spline(x1,rec1111_sam_test[3])
fitpre111e_sam1_test = make_interp_spline(x1,pre1111_sam_test[0])
fitpre111e_sam2_test = make_interp_spline(x1,pre1111_sam_test[1])
fitpre111e_sam3_test = make_interp_spline(x1,pre1111_sam_test[2])
fitpre111e_sam4_test = make_interp_spline(x1,pre1111_sam_test[3])
fit_f111e_train = make_interp_spline(x1,f1111[0])
fit_f111e_test = make_interp_spline(x1,f1111[1])
fit_f222e_train = make_interp_spline(x1,f2222[0])
fit_f222e_test = make_interp_spline(x1,f2222[1])
fit_f333e_train = make_interp_spline(x1,f3333[0])
fit_f333e_test = make_interp_spline(x1,f3333[1])
fit_f444e_train = make_interp_spline(x1,f4444[0])
fit_f444e_test = make_interp_spline(x1,f4444[1])
fit_weight111e = make_interp_spline(x1,weight1111[0])
fit_weight222e = make_interp_spline(x1,weight1111[1])
fit_weight333e = make_interp_spline(x1,weight1111[2])
fit_weight_gra111e = make_interp_spline(x2,weight1111_gra[0])
fit_weight_gra222e = make_interp_spline(x2,weight1111_gra[1])
fit_weight_gra333e = make_interp_spline(x2,weight1111_gra[2])
ys111e = fitloss111e(xs)
ys222e = fitloss222e(xs)
ys333e = fitacc111e(xs)
ys444e = fitacc222e(xs)
ys555e = fitrec111e_sam1_train(xs)
ys666e = fitrec111e_sam2_train(xs)
ys777e = fitrec111e_sam3_train(xs)
ys888e = fitrec111e_sam4_train(xs)
ys999e = fitpre111e_sam1_train(xs)
ys1000e = fitpre111e_sam2_train(xs)
ys1111e = fitpre111e_sam3_train(xs)
ys1222e = fitpre111e_sam4_train(xs)
ys1333e = fitrec111e_sam1_test(xs)
ys1444e = fitrec111e_sam2_test(xs)
ys1555e = fitrec111e_sam3_test(xs)
ys1666e = fitrec111e_sam4_test(xs)
ys1777e = fitpre111e_sam1_test(xs)
ys1888e = fitpre111e_sam2_test(xs)
ys1999e = fitpre111e_sam3_test(xs)
ys2000e = fitpre111e_sam4_test(xs)
ys2111e = fit_f111e_train(xs)
ys2222e = fit_f111e_test(xs)
ys2333e = fit_f222e_train(xs)
ys2444e = fit_f222e_test(xs)
ys2555e = fit_f333e_train(xs)
ys2666e = fit_f333e_test(xs)
ys2777e = fit_f444e_train(xs)
ys2888e = fit_f444e_test(xs)
ys2999e = fit_weight111e(xs)
ys3000e = fit_weight222e(xs)
ys3111e = fit_weight333e(xs)
ys3222e = fit_weight_gra111e(xss)
ys3333e = fit_weight_gra222e(xss)
ys3444e = fit_weight_gra333e(xss)

In [49]:
x1 = np.linspace(1,epoch,epoch)
x2 = np.linspace(2,epoch,epoch-1)
xs = np.linspace(1,epoch,1000000)
#rec召回率，pre精确度，前面的数字代表第几类样本
fitacc1 = make_interp_spline(x1,Accbox1)
fitacc2 = make_interp_spline(x1,Accbox2)
fitacc11 = make_interp_spline(x1,Accbox11)
fitacc22 = make_interp_spline(x1,Accbox22)
fitacc111 = make_interp_spline(x1,Accbox111)
fitacc222 = make_interp_spline(x1,Accbox222)
fitacc111e = make_interp_spline(x1,Accbox1111)
fitacc222e = make_interp_spline(x1,Accbox2222)

ys3 = fitacc1(xs)
ys4 = fitacc2(xs)

ys33 = fitacc11(xs)
ys44 = fitacc22(xs)

ys333 = fitacc111(xs)
ys444 = fitacc222(xs)

ys333e = fitacc111e(xs)
ys444e = fitacc222e(xs)

In [19]:
sum1 = np.zeros(28)
a1 = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum1[0] += rec1_sam_train[0][i+100]
    sum1[1] += rec1_sam_train[1][i+100]
    sum1[2] += rec1_sam_train[2][i+100]
    sum1[3] += rec1_sam_train[3][i+100]
    sum1[4] += pre1_sam_train[0][i+100]
    sum1[5] += pre1_sam_train[1][i+100]
    sum1[6] += pre1_sam_train[2][i+100]
    sum1[7] += pre1_sam_train[3][i+100]
    sum1[8] += rec1_sam_test[0][i+100]
    sum1[9] += rec1_sam_test[1][i+100]
    sum1[10] += rec1_sam_test[2][i+100]
    sum1[11] += rec1_sam_test[3][i+100]
    sum1[12] += pre1_sam_test[0][i+100]
    sum1[13] += pre1_sam_test[1][i+100]
    sum1[14] += pre1_sam_test[2][i+100]
    sum1[15] += pre1_sam_test[3][i+100]
    sum1[16] += f1[0][i+100]
    sum1[17] += f1[1][i+100]
    sum1[18] += f2[0][i+100]
    sum1[19] += f2[1][i+100]
    sum1[20] += f3[0][i+100]
    sum1[21] += f3[1][i+100]
    sum1[22] += f4[0][i+100]
    sum1[23] += f4[1][i+100]
    sum1[24] += Totallossbox1[i+100]
    sum1[25] += Totallossbox2[i+100]
    sum1[26] += Accbox1[i+100]
    sum1[27] += Accbox2[i+100]
    
    a1[0] = np.append(a1[0],rec1_sam_train[0][i+100])
    a1[1] = np.append(a1[1],rec1_sam_train[1][i+100])
    a1[2] = np.append(a1[2],rec1_sam_train[2][i+100])
    a1[3] = np.append(a1[3],rec1_sam_train[3][i+100])
    a1[4] = np.append(a1[4],pre1_sam_train[0][i+100])
    a1[5] = np.append(a1[5],pre1_sam_train[1][i+100])
    a1[6] = np.append(a1[6],pre1_sam_train[2][i+100])
    a1[7] = np.append(a1[7],pre1_sam_train[3][i+100])
    a1[8] = np.append(a1[8],rec1_sam_test[0][i+100])
    a1[9] = np.append(a1[9],rec1_sam_test[1][i+100])
    a1[10] = np.append(a1[10],rec1_sam_test[2][i+100])
    a1[11] = np.append(a1[11],rec1_sam_test[3][i+100])
    a1[12] = np.append(a1[12],pre1_sam_test[0][i+100])
    a1[13] = np.append(a1[13],pre1_sam_test[1][i+100])
    a1[14] = np.append(a1[14],pre1_sam_test[2][i+100])
    a1[15] = np.append(a1[15],pre1_sam_test[3][i+100])
    a1[16] = np.append(a1[16],f1[0][i+100])
    a1[17] = np.append(a1[17],f1[1][i+100])
    a1[18] = np.append(a1[18],f2[0][i+100])
    a1[19] = np.append(a1[19],f2[1][i+100])
    a1[20] = np.append(a1[20],f3[0][i+100])
    a1[21] = np.append(a1[21],f3[1][i+100])
    a1[22] = np.append(a1[22],f4[0][i+100])
    a1[23] = np.append(a1[23],f4[1][i+100])
    a1[24] = np.append(a1[24],Totallossbox1[i+100])
    a1[25] = np.append(a1[25],Totallossbox2[i+100])
    a1[26] = np.append(a1[26],Accbox1[i+100])
    a1[27] = np.append(a1[27],Accbox2[i+100])
for j in range(28):
    a1[j] = np.var(a1[j])
       
for k in range(28):
    sum1[k] = sum1[k]/200    
    
    
    
    
sum11 = np.zeros(28)
a11 = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum11[0] += rec11_sam_train[0][i+100]
    sum11[1] += rec11_sam_train[1][i+100]
    sum11[2] += rec11_sam_train[2][i+100]
    sum11[3] += rec11_sam_train[3][i+100]
    sum11[4] += pre11_sam_train[0][i+100]
    sum11[5] += pre11_sam_train[1][i+100]
    sum11[6] += pre11_sam_train[2][i+100]
    sum11[7] += pre11_sam_train[3][i+100]
    sum11[8] += rec11_sam_test[0][i+100]
    sum11[9] += rec11_sam_test[1][i+100]
    sum11[10] += rec11_sam_test[2][i+100]
    sum11[11] += rec11_sam_test[3][i+100]
    sum11[12] += pre11_sam_test[0][i+100]
    sum11[13] += pre11_sam_test[1][i+100]
    sum11[14] += pre11_sam_test[2][i+100]
    sum11[15] += pre11_sam_test[3][i+100]
    sum11[16] += f11[0][i+100]
    sum11[17] += f11[1][i+100]
    sum11[18] += f22[0][i+100]
    sum11[19] += f22[1][i+100]
    sum11[20] += f33[0][i+100]
    sum11[21] += f33[1][i+100]
    sum11[22] += f44[0][i+100]
    sum11[23] += f44[1][i+100]
    sum11[24] += Totallossbox11[i+100]
    sum11[25] += Totallossbox22[i+100]
    sum11[26] += Accbox11[i+100]
    sum11[27] += Accbox22[i+100]
    
    a11[0] = np.append(a11[0],rec11_sam_train[0][i+100])
    a11[1] = np.append(a11[1],rec11_sam_train[1][i+100])
    a11[2] = np.append(a11[2],rec11_sam_train[2][i+100])
    a11[3] = np.append(a11[3],rec11_sam_train[3][i+100])
    a11[4] = np.append(a11[4],pre11_sam_train[0][i+100])
    a11[5] = np.append(a11[5],pre11_sam_train[1][i+100])
    a11[6] = np.append(a11[6],pre11_sam_train[2][i+100])
    a11[7] = np.append(a11[7],pre11_sam_train[3][i+100])
    a11[8] = np.append(a11[8],rec11_sam_test[0][i+100])
    a11[9] = np.append(a11[9],rec11_sam_test[1][i+100])
    a11[10] = np.append(a11[10],rec11_sam_test[2][i+100])
    a11[11] = np.append(a11[11],rec11_sam_test[3][i+100])
    a11[12] = np.append(a11[12],pre11_sam_test[0][i+100])
    a11[13] = np.append(a11[13],pre11_sam_test[1][i+100])
    a11[14] = np.append(a11[14],pre11_sam_test[2][i+100])
    a11[15] = np.append(a11[15],pre11_sam_test[3][i+100])
    a11[16] = np.append(a11[16],f11[0][i+100])
    a11[17] = np.append(a11[17],f11[1][i+100])
    a11[18] = np.append(a11[18],f22[0][i+100])
    a11[19] = np.append(a11[19],f22[1][i+100])
    a11[20] = np.append(a11[20],f33[0][i+100])
    a11[21] = np.append(a11[21],f33[1][i+100])
    a11[22] = np.append(a11[22],f44[0][i+100])
    a11[23] = np.append(a11[23],f44[1][i+100])
    a11[24] = np.append(a11[24],Totallossbox11[i+100])
    a11[25] = np.append(a11[25],Totallossbox22[i+100])
    a11[26] = np.append(a11[26],Accbox11[i+100])
    a11[27] = np.append(a11[27],Accbox22[i+100])
    
for j in range(28):
    a11[j] = np.var(a11[j])
    
for k in range(28):
    sum11[k] = sum11[k]/200
    

    

sum111 = np.zeros(28)
a111 = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum111[0] += rec111_sam_train[0][i+100]
    sum111[1] += rec111_sam_train[1][i+100]
    sum111[2] += rec111_sam_train[2][i+100]
    sum111[3] += rec111_sam_train[3][i+100]
    sum111[4] += pre111_sam_train[0][i+100]
    sum111[5] += pre111_sam_train[1][i+100]
    sum111[6] += pre111_sam_train[2][i+100]
    sum111[7] += pre111_sam_train[3][i+100]
    sum111[8] += rec111_sam_test[0][i+100]
    sum111[9] += rec111_sam_test[1][i+100]
    sum111[10] += rec111_sam_test[2][i+100]
    sum111[11] += rec111_sam_test[3][i+100]
    sum111[12] += pre111_sam_test[0][i+100]
    sum111[13] += pre111_sam_test[1][i+100]
    sum111[14] += pre111_sam_test[2][i+100]
    sum111[15] += pre111_sam_test[3][i+100]
    sum111[16] += f111[0][i+100]
    sum111[17] += f111[1][i+100]
    sum111[18] += f222[0][i+100]
    sum111[19] += f222[1][i+100]
    sum111[20] += f333[0][i+100]
    sum111[21] += f333[1][i+100]
    sum111[22] += f444[0][i+100]
    sum111[23] += f444[1][i+100]
    sum111[24] += Totallossbox111[i+100]
    sum111[25] += Totallossbox222[i+100]
    sum111[26] += Accbox111[i+100]
    sum111[27] += Accbox222[i+100]
    
    a111[0] = np.append(a111[0],rec111_sam_train[0][i+100])
    a111[1] = np.append(a111[1],rec111_sam_train[1][i+100])
    a111[2] = np.append(a111[2],rec111_sam_train[2][i+100])
    a111[3] = np.append(a111[3],rec111_sam_train[3][i+100])
    a111[4] = np.append(a111[4],pre111_sam_train[0][i+100])
    a111[5] = np.append(a111[5],pre111_sam_train[1][i+100])
    a111[6] = np.append(a111[6],pre111_sam_train[2][i+100])
    a111[7] = np.append(a111[7],pre111_sam_train[3][i+100])
    a111[8] = np.append(a111[8],rec111_sam_test[0][i+100])
    a111[9] = np.append(a111[9],rec111_sam_test[1][i+100])
    a111[10] = np.append(a111[10],rec111_sam_test[2][i+100])
    a111[11] = np.append(a111[11],rec111_sam_test[3][i+100])
    a111[12] = np.append(a111[12],pre111_sam_test[0][i+100])
    a111[13] = np.append(a111[13],pre111_sam_test[1][i+100])
    a111[14] = np.append(a111[14],pre111_sam_test[2][i+100])
    a111[15] = np.append(a111[15],pre111_sam_test[3][i+100])
    a111[16] = np.append(a111[16],f111[0][i+100])
    a111[17] = np.append(a111[17],f111[1][i+100])
    a111[18] = np.append(a111[18],f222[0][i+100])
    a111[19] = np.append(a111[19],f222[1][i+100])
    a111[20] = np.append(a111[20],f333[0][i+100])
    a111[21] = np.append(a111[21],f333[1][i+100])
    a111[22] = np.append(a111[22],f444[0][i+100])
    a111[23] = np.append(a111[23],f444[1][i+100])
    a111[24] = np.append(a111[24],Totallossbox111[i+100])
    a111[25] = np.append(a111[25],Totallossbox222[i+100])
    a111[26] = np.append(a111[26],Accbox111[i+100])
    a111[27] = np.append(a111[27],Accbox222[i+100])
    
for j in range(28):
    a111[j] = np.var(a111[j])
for k in range(28):
    sum111[k] = sum111[k]/200    
    


sum111e = np.zeros(28)
a111e = [[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]] #28个

for i in range(200):
    sum111e[0] += rec1111_sam_train[0][i+100]
    sum111e[1] += rec1111_sam_train[1][i+100]
    sum111e[2] += rec1111_sam_train[2][i+100]
    sum111e[3] += rec1111_sam_train[3][i+100]
    sum111e[4] += pre1111_sam_train[0][i+100]
    sum111e[5] += pre1111_sam_train[1][i+100]
    sum111e[6] += pre1111_sam_train[2][i+100]
    sum111e[7] += pre1111_sam_train[3][i+100]
    sum111e[8] += rec1111_sam_test[0][i+100]
    sum111e[9] += rec1111_sam_test[1][i+100]
    sum111e[10] += rec1111_sam_test[2][i+100]
    sum111e[11] += rec1111_sam_test[3][i+100]
    sum111e[12] += pre1111_sam_test[0][i+100]
    sum111e[13] += pre1111_sam_test[1][i+100]
    sum111e[14] += pre1111_sam_test[2][i+100]
    sum111e[15] += pre1111_sam_test[3][i+100]
    sum111e[16] += f1111[0][i+100]
    sum111e[17] += f1111[1][i+100]
    sum111e[18] += f2222[0][i+100]
    sum111e[19] += f2222[1][i+100]
    sum111e[20] += f3333[0][i+100]
    sum111e[21] += f3333[1][i+100]
    sum111e[22] += f4444[0][i+100]
    sum111e[23] += f4444[1][i+100]
    sum111e[24] += Totallossbox1111[i+100]
    sum111e[25] += Totallossbox2222[i+100]
    sum111e[26] += Accbox1111[i+100]
    sum111e[27] += Accbox2222[i+100]
    
    a111e[0] = np.append(a111e[0],rec1111_sam_train[0][i+100])
    a111e[1] = np.append(a111e[1],rec1111_sam_train[1][i+100])
    a111e[2] = np.append(a111e[2],rec1111_sam_train[2][i+100])
    a111e[3] = np.append(a111e[3],rec1111_sam_train[3][i+100])
    a111e[4] = np.append(a111e[4],pre1111_sam_train[0][i+100])
    a111e[5] = np.append(a111e[5],pre1111_sam_train[1][i+100])
    a111e[6] = np.append(a111e[6],pre1111_sam_train[2][i+100])
    a111e[7] = np.append(a111e[7],pre1111_sam_train[3][i+100])
    a111e[8] = np.append(a111e[8],rec1111_sam_test[0][i+100])
    a111e[9] = np.append(a111e[9],rec1111_sam_test[1][i+100])
    a111e[10] = np.append(a111e[10],rec1111_sam_test[2][i+100])
    a111e[11] = np.append(a111e[11],rec1111_sam_test[3][i+100])
    a111e[12] = np.append(a111e[12],pre1111_sam_test[0][i+100])
    a111e[13] = np.append(a111e[13],pre1111_sam_test[1][i+100])
    a111e[14] = np.append(a111e[14],pre1111_sam_test[2][i+100])
    a111e[15] = np.append(a111e[15],pre1111_sam_test[3][i+100])
    a111e[16] = np.append(a111e[16],f1111[0][i+100])
    a111e[17] = np.append(a111e[17],f1111[1][i+100])
    a111e[18] = np.append(a111e[18],f2222[0][i+100])
    a111e[19] = np.append(a111e[19],f2222[1][i+100])
    a111e[20] = np.append(a111e[20],f3333[0][i+100])
    a111e[21] = np.append(a111e[21],f3333[1][i+100])
    a111e[22] = np.append(a111e[22],f4444[0][i+100])
    a111e[23] = np.append(a111e[23],f4444[1][i+100])
    a111e[24] = np.append(a111e[24],Totallossbox1111[i+100])
    a111e[25] = np.append(a111e[25],Totallossbox2222[i+100])
    a111e[26] = np.append(a111e[26],Accbox1111[i+100])
    a111e[27] = np.append(a111e[27],Accbox2222[i+100])
    
for j in range(28):
    a111e[j] = np.var(a111e[j])
for k in range(28):
    sum111e[k] = sum111e[k]/200 

In [20]:
with open('cifar100.txt', 'a') as f:
    lines = ['4种神经网络P-R曲线面积为:',' ',str(pr_aucQRCNN),',',' ',str(pr_aucCNN1),',',' ',str(pr_aucQDCNN),',',' ',str(pr_aucCNN2),'\n'
            '4种神经网络ROC曲线面积为:',' ',str(roc_aucQRCNN),',',' ',str(roc_aucCNN1),',',' ',str(roc_aucQDCNN),',',' ',str(roc_aucCNN2),'\n','\n',
             '第一组神经网络的样本0train的recall方差:',' ',str(a1[0]),'\n',
             '第一组神经网络的样本1train的recall方差:',' ',str(a1[1]),'\n',
             '第一组神经网络的样本2train的recall方差:',' ',str(a1[2]),'\n',
             '第一组神经网络的样本3train的recall方差:',' ',str(a1[3]),'\n',
             '第一组神经网络的样本0train的precision方差:',' ',str(a1[4]),'\n',
             '第一组神经网络的样本1train的precision方差:',' ',str(a1[5]),'\n',
             '第一组神经网络的样本2train的precision方差:',' ',str(a1[6]),'\n',
             '第一组神经网络的样本3train的precision方差:',' ',str(a1[7]),'\n',
             '第一组神经网络的样本0test的recall方差:',' ',str(a1[8]),'\n',
             '第一组神经网络的样本1test的recall方差:',' ',str(a1[9]),'\n',
             '第一组神经网络的样本2test的recall方差:',' ',str(a1[10]),'\n',
             '第一组神经网络的样本3test的recall方差:',' ',str(a1[11]),'\n',
             '第一组神经网络的样本0test的precision方差:',' ',str(a1[12]),'\n',
             '第一组神经网络的样本1test的precision方差:',' ',str(a1[13]),'\n',
             '第一组神经网络的样本2test的precision方差:',' ',str(a1[14]),'\n',
             '第一组神经网络的样本3test的precision方差:',' ',str(a1[15]),'\n',
             '第一组神经网络的样本0train的F1-score方差:',' ',str(a1[16]),'\n',
             '第一组神经网络的样本0test的F1-score方差:',' ',str(a1[17]),'\n',
             '第一组神经网络的样本1train的F1-score方差:',' ',str(a1[18]),'\n',
             '第一组神经网络的样本1test的F1-score方差:',' ',str(a1[19]),'\n',
             '第一组神经网络的样本2train的F1-score方差:',' ',str(a1[20]),'\n',
             '第一组神经网络的样本2test的F1-score方差:',' ',str(a1[21]),'\n',
             '第一组神经网络的样本3train的F1-score方差:',' ',str(a1[22]),'\n',
             '第一组神经网络的样本3test的F1-score方差:',' ',str(a1[23]),'\n',
             "第一组神经网络的loss的train方差:",' ',str(a1[24]),'\n',
             "第一组神经网络的loss的test方差:",' ',str(a1[25]),'\n',
             "第一组神经网络的acc的train方差:",' ',str(a1[26]),'\n',
             "第一组神经网络的acc的test方差:",' ',str(a1[27]),'\n',
             
             '第一组神经网络的样本0train的recall均值:',' ',str(sum1[0]),'\n',
            '第一组神经网络的样本1train的recall均值:',' ',str(sum1[1]),'\n',
            '第一组神经网络的样本2train的recall均值:',' ',str(sum1[2]),'\n',
             '第一组神经网络的样本3train的recall均值:',' ',str(sum1[3]),'\n',
            '第一组神经网络的样本0train的precision均值:',' ',str(sum1[4]),'\n',
            '第一组神经网络的样本1train的precision均值:',' ',str(sum1[5]),'\n',
            '第一组神经网络的样本2train的precision均值:',' ',str(sum1[6]),'\n',
             '第一组神经网络的样本3train的precision均值:',' ',str(sum1[7]),'\n',
            '第一组神经网络的样本0test的recall均值:',' ',str(sum1[8]),'\n',
            '第一组神经网络的样本1test的recall均值:',' ',str(sum1[9]),'\n',
            '第一组神经网络的样本2test的recall均值:',' ',str(sum1[10]),'\n',
             '第一组神经网络的样本3test的recall均值:',' ',str(sum1[11]),'\n',
            '第一组神经网络的样本0test的precision均值:',' ',str(sum1[12]),'\n',
            '第一组神经网络的样本1test的precision均值:',' ',str(sum1[13]),'\n',
            '第一组神经网络的样本2test的precision均值:',' ',str(sum1[14]),'\n',
             '第一组神经网络的样本3test的precision均值:',' ',str(sum1[15]),'\n',
            '第一组神经网络的样本0train的F1-score均值:',' ',str(sum1[16]),'\n',
            '第一组神经网络的样本0test的F1-score均值:',' ',str(sum1[17]),'\n',
            '第一组神经网络的样本1train的F1-score均值:',' ',str(sum1[18]),'\n',
            '第一组神经网络的样本1test的F1-score均值:',' ',str(sum1[19]),'\n',
            '第一组神经网络的样本2train的F1-score均值:',' ',str(sum1[20]),'\n',
            '第一组神经网络的样本2test的F1-score均值:',' ',str(sum1[21]),'\n',
             '第一组神经网络的样本3train的F1-score均值:',' ',str(sum1[22]),'\n',
            '第一组神经网络的样本3test的F1-score均值:',' ',str(sum1[23]),'\n',
            "第一组神经网络的loss的train均值:",' ',str(sum1[24]),'\n',
            "第一组神经网络的loss的test均值:",' ',str(sum1[25]),'\n',
            "第一组神经网络的acc的train均值:",' ',str(sum1[26]),'\n',
            "第一组神经网络的acc的test均值:",' ',str(sum1[27]),'\n',
             
             '第2组神经网络的样本0train的recall方差:',' ',str(a11[0]),'\n',   
            '第2组神经网络的样本1train的recall方差:',' ',str(a11[1]),'\n', 
            '第2组神经网络的样本2train的recall方差:',' ',str(a11[2]),'\n', 
             '第2组神经网络的样本3train的recall方差:',' ',str(a11[3]),'\n',
            '第2组神经网络的样本0train的precision方差:',' ',str(a11[4]),'\n', 
            '第2组神经网络的样本1train的precision方差:',' ',str(a11[5]),'\n', 
            '第2组神经网络的样本2train的precision方差:',' ',str(a11[6]),'\n', 
             '第2组神经网络的样本3train的precision方差:',' ',str(a11[7]),'\n',
            '第2组神经网络的样本0test的recall方差:',' ',str(a11[8]),'\n', 
            '第2组神经网络的样本1test的recall方差:',' ',str(a11[9]),'\n', 
            '第2组神经网络的样本2test的recall方差:',' ',str(a11[10]),'\n',
             '第2组神经网络的样本3test的recall方差:',' ',str(a11[11]),'\n',
            '第2组神经网络的样本0test的precision方差:',' ',str(a11[12]),'\n', 
            '第2组神经网络的样本1test的precision方差:',' ',str(a11[13]),'\n', 
            '第2组神经网络的样本2test的precision方差:',' ',str(a11[14]),'\n', 
             '第2组神经网络的样本3test的precision方差:',' ',str(a11[15]),'\n',
            '第2组神经网络的样本0train的F1-score方差:',' ',str(a11[16]),'\n', 
            '第2组神经网络的样本0test的F1-score方差:',' ',str(a11[17]),'\n', 
            '第2组神经网络的样本1train的F1-score方差:',' ',str(a11[18]),'\n', 
            '第2组神经网络的样本1test的F1-score方差:',' ',str(a11[19]),'\n', 
            '第2组神经网络的样本2train的F1-score方差:',' ',str(a11[20]),'\n', 
            '第2组神经网络的样本2test的F1-score方差:',' ',str(a11[21]),'\n', 
             '第2组神经网络的样本3train的F1-score方差:',' ',str(a11[22]),'\n', 
            '第2组神经网络的样本3test的F1-score方差:',' ',str(a11[23]),'\n'
            "第2组神经网络的loss的train方差:",' ',str(a11[24]),'\n', 
            "第2组神经网络的loss的test方差:",' ',str(a11[25]),'\n', 
            "第2组神经网络的acc的train方差:",' ',str(a11[26]),'\n', 
            "第2组神经网络的acc的test方差:",' ',str(a11[27]),'\n', 
             
             '第2组神经网络的样本0train的recall均值:',' ',str(sum11[0]),'\n',  
            '第2组神经网络的样本1train的recall均值:',' ',str(sum11[1]),'\n',  
            '第2组神经网络的样本2train的recall均值:',' ',str(sum11[2]),'\n',
              '第2组神经网络的样本3train的recall均值:',' ',str(sum11[3]),'\n',
            '第2组神经网络的样本0train的precision均值:',' ',str(sum11[4]),'\n',  
            '第2组神经网络的样本1train的precision均值:',' ',str(sum11[5]),'\n',  
            '第2组神经网络的样本2train的precision均值:',' ',str(sum11[6]),'\n',
             '第2组神经网络的样本3train的precision均值:',' ',str(sum11[7]),'\n',
            '第2组神经网络的样本0test的recall均值:',' ',str(sum11[8]),'\n',  
            '第2组神经网络的样本1test的recall均值:',' ',str(sum11[9]),'\n',  
            '第2组神经网络的样本2test的recall均值:',' ',str(sum11[10]),'\n',
             '第2组神经网络的样本3test的recall均值:',' ',str(sum11[11]),'\n',
            '第2组神经网络的样本0test的precision均值:',' ',str(sum11[12]),'\n',  
            '第2组神经网络的样本1test的precision均值:',' ',str(sum11[13]),'\n',  
            '第2组神经网络的样本2test的precision均值:',' ',str(sum11[14]),'\n',
             '第2组神经网络的样本3test的precision均值:',' ',str(sum11[15]),'\n',
            '第2组神经网络的样本0train的F1-score均值:',' ',str(sum11[16]),'\n',  
            '第2组神经网络的样本0test的F1-score均值:',' ',str(sum11[17]),'\n',  
            '第2组神经网络的样本1train的F1-score均值:',' ',str(sum11[18]),'\n',  
            '第2组神经网络的样本1test的F1-score均值:',' ',str(sum11[19]),'\n',  
            '第2组神经网络的样本2train的F1-score均值:',' ',str(sum11[20]),'\n',  
            '第2组神经网络的样本2test的F1-score均值:',' ',str(sum11[21]),'\n',
             '第2组神经网络的样本3train的F1-score均值:',' ',str(sum11[22]),'\n',  
            '第2组神经网络的样本3test的F1-score均值:',' ',str(sum11[23]),'\n',
            "第2组神经网络的loss的train均值:",' ',str(sum11[24]),'\n',  
            "第2组神经网络的loss的test均值:",' ',str(sum11[25]),'\n',  
            "第2组神经网络的acc的train均值:",' ',str(sum11[26]),'\n',  
            "第2组神经网络的acc的test均值:",' ',str(sum11[27]),'\n', 
             
             
             '第3组神经网络的样本0train的recall方差:',' ',str(a111[0]),'\n',
            '第3组神经网络的样本1train的recall方差:',' ',str(a111[1]),'\n',
            '第3组神经网络的样本2train的recall方差:',' ',str(a111[2]),'\n',
            '第3组神经网络的样本3train的recall方差:',' ',str(a111[3]),'\n',
            '第3组神经网络的样本0train的precision方差:',' ',str(a111[4]),'\n',
            '第3组神经网络的样本1train的precision方差:',' ',str(a111[5]),'\n',
            '第3组神经网络的样本2train的precision方差:',' ',str(a111[6]),'\n',
            '第3组神经网络的样本3train的precision方差:',' ',str(a111[7]),'\n',
            '第3组神经网络的样本0test的recall方差:',' ',str(a111[8]),'\n',
            '第3组神经网络的样本1test的recall方差:',' ',str(a111[9]),'\n',
            '第3组神经网络的样本2test的recall方差:',' ',str(a111[10]),'\n',
            '第3组神经网络的样本3test的recall方差:',' ',str(a111[11]),'\n',
            '第3组神经网络的样本0test的precision方差:',' ',str(a111[12]),'\n',
            '第3组神经网络的样本1test的precision方差:',' ',str(a111[13]),'\n',
            '第3组神经网络的样本2test的precision方差:',' ',str(a111[14]),'\n',
            '第3组神经网络的样本3test的precision方差:',' ',str(a111[15]),'\n',
            '第3组神经网络的样本0train的F1-score方差:',' ',str(a111[16]),'\n',
            '第3组神经网络的样本1train的F1-score方差:',' ',str(a111[17]),'\n',
            '第3组神经网络的样本2train的F1-score方差:',' ',str(a111[18]),'\n',
            '第3组神经网络的样本3train的F1-score方差:',' ',str(a111[19]),'\n',
            '第3组神经网络的样本0test的F1-score方差:',' ',str(a111[20]),'\n',
            '第3组神经网络的样本1test的F1-score方差:',' ',str(a111[21]),'\n',
            '第3组神经网络的样本2test的F1-score方差:',' ',str(a111[22]),'\n',
            '第3组神经网络的样本3test的F1-score方差:',' ',str(a111[23]),'\n',
            "第3组神经网络的loss的train方差:",' ',str(a111[24]),'\n',
            "第3组神经网络的loss的test方差:",' ',str(a111[25]),'\n',
            "第3组神经网络的acc的train方差:",' ',str(a111[26]),'\n',
            "第3组神经网络的acc的test方差:",' ',str(a111[27]),'\n',
             
             '第3组神经网络的样本0train的recall均值:',' ',str(sum111[0]),'\n',
            '第3组神经网络的样本1train的recall均值:',' ',str(sum111[1]),'\n',
            '第3组神经网络的样本2train的recall均值:',' ',str(sum111[2]),'\n',
            '第3组神经网络的样本3train的recall均值:',' ',str(sum111[3]),'\n',
            '第3组神经网络的样本0train的precision均值:',' ',str(sum111[4]),'\n',
            '第3组神经网络的样本1train的precision均值:',' ',str(sum111[5]),'\n',
            '第3组神经网络的样本2train的precision均值:',' ',str(sum111[6]),'\n',
            '第3组神经网络的样本3train的precision均值:',' ',str(sum111[7]),'\n',
            '第3组神经网络的样本0test的recall均值:',' ',str(sum111[8]),'\n',
            '第3组神经网络的样本1test的recall均值:',' ',str(sum111[9]),'\n',
            '第3组神经网络的样本2test的recall均值:',' ',str(sum111[10]),'\n',
            '第3组神经网络的样本3test的recall均值:',' ',str(sum111[11]),'\n',
            '第3组神经网络的样本0test的precision均值:',' ',str(sum111[12]),'\n',
            '第3组神经网络的样本1test的precision均值:',' ',str(sum111[13]),'\n',
            '第3组神经网络的样本2test的precision均值:',' ',str(sum111[14]),'\n',
            '第3组神经网络的样本3test的precision均值:',' ',str(sum111[15]),'\n',
            '第3组神经网络的样本0train的F1-score均值:',' ',str(sum111[16]),'\n',
            '第3组神经网络的样本1train的F1-score均值:',' ',str(sum111[17]),'\n',
            '第3组神经网络的样本2train的F1-score均值:',' ',str(sum111[18]),'\n',
            '第3组神经网络的样本3train的F1-score均值:',' ',str(sum111[19]),'\n',
            '第3组神经网络的样本0test的F1-score均值:',' ',str(sum111[20]),'\n',
            '第3组神经网络的样本1test的F1-score均值:',' ',str(sum111[21]),'\n',
            '第3组神经网络的样本2test的F1-score均值:',' ',str(sum111[22]),'\n',
            '第3组神经网络的样本3test的F1-score均值:',' ',str(sum111[23]),'\n',
            "第3组神经网络的loss的train均值:",' ',str(sum111[24]),'\n',
            "第3组神经网络的loss的test均值:",' ',str(sum111[25]),'\n',
            "第3组神经网络的acc的train均值:",' ',str(sum111[26]),'\n',
            "第3组神经网络的acc的test均值:",' ',str(sum111[27]),'\n',
             
             
            '第4组神经网络的样本0train的recall方差:',' ',str(a111e[0]),'\n',
            '第4组神经网络的样本1train的recall方差:',' ',str(a111e[1]),'\n',
            '第4组神经网络的样本2train的recall方差:',' ',str(a111e[2]),'\n',
             '第4组神经网络的样本3train的recall方差:',' ',str(a111e[3]),'\n',
            '第4组神经网络的样本0train的precision方差:',' ',str(a111e[4]),'\n',
            '第4组神经网络的样本1train的precision方差:',' ',str(a111e[5]),'\n',
            '第4组神经网络的样本2train的precision方差:',' ',str(a111e[6]),'\n',
             '第4组神经网络的样本3train的precision方差:',' ',str(a111e[7]),'\n',
            '第4组神经网络的样本0test的recall方差:',' ',str(a111e[8]),'\n',
            '第4组神经网络的样本1test的recall方差:',' ',str(a111e[9]),'\n',
            '第4组神经网络的样本2test的recall方差:',' ',str(a111e[10]),'\n',
             '第4组神经网络的样本3test的recall方差:',' ',str(a111e[11]),'\n',
            '第4组神经网络的样本0test的precision方差:',' ',str(a111e[12]),'\n',
            '第4组神经网络的样本1test的precision方差:',' ',str(a111e[13]),'\n',
            '第4组神经网络的样本2test的precision方差:',' ',str(a111e[14]),'\n',
              '第4组神经网络的样本3test的precision方差:',' ',str(a111e[15]),'\n',
            '第4组神经网络的样本0train的F1-score方差:',' ',str(a111e[16]),'\n',
            '第4组神经网络的样本0test的F1-score方差:',' ',str(a111e[17]),'\n',
            '第4组神经网络的样本1train的F1-score方差:',' ',str(a111e[18]),'\n',
            '第4组神经网络的样本1test的F1-score方差:',' ',str(a111e[19]),'\n',
            '第4组神经网络的样本2train的F1-score方差:',' ',str(a111e[20]),'\n',
            '第4组神经网络的样本2test的F1-score方差:',' ',str(a111e[21]),'\n',
             '第4组神经网络的样本3train的F1-score方差:',' ',str(a111e[22]),'\n',
            '第4组神经网络的样本3test的F1-score方差:',' ',str(a111e[23]),'\n',
            "第4组神经网络的loss的train方差:",' ',str(a111e[24]),'\n',
            "第4组神经网络的loss的test方差:",' ',str(a111e[25]),'\n',
            "第4组神经网络的acc的train方差:",' ',str(a111e[26]),'\n',
            "第4组神经网络的acc的test方差:",' ',str(a111e[27]),'\n',
             
             '第4组神经网络的样本0train的recall均值:',' ',str(sum111e[0]),'\n',
            '第4组神经网络的样本1train的recall均值:',' ',str(sum111e[1]),'\n',
            '第4组神经网络的样本2train的recall均值:',' ',str(sum111e[2]),'\n',
             '第4组神经网络的样本3train的recall均值:',' ',str(sum111e[3]),'\n',
            '第4组神经网络的样本0train的precision均值:',' ',str(sum111e[4]),'\n',
            '第4组神经网络的样本1train的precision均值:',' ',str(sum111e[5]),'\n',
            '第4组神经网络的样本2train的precision均值:',' ',str(sum111e[6]),'\n',
             '第4组神经网络的样本3train的precision均值:',' ',str(sum111e[7]),'\n',
            '第4组神经网络的样本0test的recall均值:',' ',str(sum111e[8]),'\n',
            '第4组神经网络的样本1test的recall均值:',' ',str(sum111e[9]),'\n',
            '第4组神经网络的样本2test的recall均值:',' ',str(sum111e[10]),'\n',
             '第4组神经网络的样本3test的recall均值:',' ',str(sum111e[11]),'\n',
            '第4组神经网络的样本0test的precision均值:',' ',str(sum111e[12]),'\n',
            '第4组神经网络的样本1test的precision均值:',' ',str(sum111e[13]),'\n',
            '第4组神经网络的样本2test的precision均值:',' ',str(sum111e[14]),'\n',
             '第4组神经网络的样本3test的precision均值:',' ',str(sum111e[15]),'\n',
            '第4组神经网络的样本0train的F1-score均值:',' ',str(sum111e[16]),'\n',
            '第4组神经网络的样本0test的F1-score均值:',' ',str(sum111e[17]),'\n',
            '第4组神经网络的样本1train的F1-score均值:',' ',str(sum111e[18]),'\n',
            '第4组神经网络的样本1test的F1-score均值:',' ',str(sum111e[19]),'\n',
            '第4组神经网络的样本2train的F1-score均值:',' ',str(sum111e[20]),'\n',
            '第4组神经网络的样本2test的F1-score均值:',' ',str(sum111e[21]),'\n',
             '第4组神经网络的样本3train的F1-score均值:',' ',str(sum111e[22]),'\n',
            '第4组神经网络的样本3test的F1-score均值:',' ',str(sum111e[23]),'\n',
            "第4组神经网络的loss的train均值:",' ',str(sum111e[24]),'\n',
            "第4组神经网络的loss的test均值:",' ',str(sum111e[25]),'\n',
            "第4组神经网络的acc的train均值:",' ',str(sum111e[26]),'\n',
            "第4组神经网络的acc的test均值:",' ',str(sum111e[27]),'\n',
            ]
    f.writelines(lines)

In [50]:
%matplotlib qt5
#plt.subplot(121)
#plt.plot(xs,ys1,color = "blue",label = "training of QRCNN",lw = 3)
#plt.plot(xs,ys2,color = "red",label = "testing of QRCNN",lw = 3)
#plt.plot(xs,yss11,color = "green",label = "training of CNN(leaky-relu)",lw = 3)
#plt.plot(xs,yss22,color = "yellow",label = "testing of CNN(leaky-relu)",lw = 3)
#plt.plot(xs,ys111,color = "black",label = "training of QDCNN",lw = 3)
#plt.plot(xs,ys222,color = "orange",label = "testing of QDCNN",lw = 3)
#plt.plot(xs,ys111e,color = "grey",label = "training of CNN(rot-relu)",lw = 3)
#plt.plot(xs,ys222e,color = "purple",label = "testing of CNN(rot-relu)",lw = 3)
#plt.title("loss-epoch plot in testing set of CIFAR100")
#plt.xlabel("epoch")
#plt.ylabel("loss")
#plt.legend(loc='upper right')
#plt.grid()

#plt.subplot(122)
plt.tick_params(labelsize=24)
#plt.plot(xs,ys3,color = "blue",label = "training of QRCNN",lw = 3)
plt.plot(xs,ys4,color = "red",label = "testing of ResQNet2",lw = 3)
#plt.plot(xs,ys33,color = "green",label = "training of CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys44,color = "green",label = "testing of QRCNN",lw = 3)
#plt.plot(xs,ys333,color = "black",label = "training of QDCNN",lw = 3)
plt.plot(xs,ys444,color = "orange",label = "testing of QDCNN",lw = 3)
#plt.plot(xs,ys333e,color = "grey",label = "training of CNN(rot-relu)",lw = 3)
plt.plot(xs,ys444e,color = "purple",label = "testing of ResQNet1",lw = 3)
plt.title("accuracy-epoch plot in testing set of CIFAR100 under gaussian asymmetrical noise attack in form 2",fontsize=24)
plt.xlabel("epoch",fontsize=24)
plt.ylabel("accuracy",fontsize=24)
plt.legend(loc='lower right',fontsize=24)
plt.grid()

In [51]:
sum1 = 0
sum2 = 0
sum3 = 0
sum4 = 0
for i in range(200):
    sum1 += Accbox2[i+100]
    sum2 += Accbox22[i+100]
    sum3 += Accbox222[i+100]
    sum4 += Accbox2222[i+100]
sum1 = sum1/200
sum2 = sum2/200
sum3 = sum3/200
sum4 = sum4/200
print(sum1)
print(sum2)
print(sum3)
print(sum4)

0.7578999999999999
0.7719875000000002
0.7919750000000001
0.7741499999999998


In [53]:
err1 = 0.95*np.sqrt(sum1*(1-sum1)/400)
err2 = 0.95*np.sqrt(sum2*(1-sum2)/400)
err3 = 0.95*np.sqrt(sum3*(1-sum3)/400)
err4 = 0.95*np.sqrt(sum4*(1-sum4)/400)
print(err1)
print(err2)
print(err3)
print(err4)

0.020346839433619663
0.019928658814568047
0.01928000362655162
0.019861690776073804


In [54]:
f=open("Resqnet.txt","w")
f.writelines(str(Accbox2))
f.writelines(str(Accbox22))
f.writelines(str(Accbox222))
f.writelines(str(Accbox2222))

In [22]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys5,label = "sample 0 in training set in QRCNN",lw = 3)
plt.plot(xs,ys13,label = "sample 0 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys55,label = "sample 0 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys133,label = "sample 0 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys555,label = "sample 0 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1333,label = "sample 0 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys555e,label = "sample 0 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1333e,label = "sample 0 in testing set in CNN(rot-relu)",lw = 3)
plt.title("recall-epoch plot of CIFAR100 of sample 0")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xs,ys6,label = "sample 1 in training set in QRCNN",lw = 3)
plt.plot(xs,ys14,label = "sample 1 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys66,label = "sample 1 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys144,label = "sample 1 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys666,label = "sample 1 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1444,label = "sample 1 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys666e,label = "sample 1 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1444e,label = "sample 1 in testing set in CNN(rot-relu)",lw = 3)
plt.title("recall-epoch plot of CIFAR100 of sample 1")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(223)
plt.plot(xs,ys7,label = "sample 2 in training set in QRCNN",lw = 3)
plt.plot(xs,ys15,label = "sample 2 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys77,label = "sample 2 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys155,label = "sample 2 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys777,label = "sample 2 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1555,label = "sample 2 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys777e,label = "sample 2 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1555e,label = "sample 2 in testing set in CNN(rot-relu)",lw = 3)
plt.title("recall-epoch plot of CIFAR100 of sample 2")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(224)
plt.plot(xs,ys8,label = "sample 3 in training set in QRCNN",lw = 3)
plt.plot(xs,ys16,label = "sample 3 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys88,label = "sample 3 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys166,label = "sample 3 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys888,label = "sample 3 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1666,label = "sample 3 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys888e,label = "sample 3 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1666e,label = "sample 3 in testing set in CNN(rot-relu)",lw = 3)
plt.title("recall-epoch plot of CIFAR100 of sample 3")
plt.xlabel("epoch")
plt.ylabel("recall")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

In [23]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys9,label = "sample 0 in training set in QRCNN",lw = 3)
plt.plot(xs,ys17,label = "sample 0 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys99,label = "sample 0 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys177,label = "sample 0 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys999,label = "sample 0 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1777,label = "sample 0 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys999e,label = "sample 0 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1777e,label = "sample 0 in testing set in CNN(rot-relu)",lw = 3)
plt.title("precision-epoch plot of CIFAR100 of sample 0")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xs,ys10,label = "sample 1 in training set in QRCNN",lw = 3)
plt.plot(xs,ys18,label = "sample 1 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys100,label = "sample 1 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys188,label = "sample 1 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1000,label = "sample 1 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1888,label = "sample 1 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys1000e,label = "sample 1 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1888e,label = "sample 1 in testing set in CNN(rot-relu)",lw = 3)
plt.title("precision-epoch plot of CIFAR100 of sample 1")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(223)
plt.plot(xs,ys11,label = "sample 2 in training set in QRCNN",lw = 3) 
plt.plot(xs,ys19,label = "sample 2 in testing set in QRCNN",lw = 3)
plt.plot(xs,yss111,label = "sample 2 in training set in CNN(leaky-relu)",lw = 3)    
plt.plot(xs,ys199,label = "sample 2 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1111,label = "sample 2 in training set in QDCNN",lw = 3)
plt.plot(xs,ys1999,label = "sample 2 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys1111e,label = "sample 2 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys1999e,label = "sample 2 in testing set in CNN(rot-relu)",lw = 3)
plt.title("precision-epoch plot of CIFAR100 of sample 2")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(224)
plt.plot(xs,ys12,label = "sample 3 in training set in QRCNN",lw = 3)
plt.plot(xs,ys20,label = "sample 3 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys122,label = "sample 3 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys200,label = "sample 3 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys1222,label = "sample 3 in training set in QDCNN",lw = 3)
plt.plot(xs,ys2000,label = "sample 3 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys1222e,label = "sample 3 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys2000e,label = "sample 3 in testing set in CNN(rot-relu)",lw = 3)
plt.title("precision-epoch plot of CIFAR100 of sample 3")
plt.xlabel("epoch")
plt.ylabel("precision")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

In [24]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys21,label = "sample 0 in training set in QRCNN",lw = 3)
plt.plot(xs,ys22,label = "sample 0 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys211,label = "sample 0 in training set in CNN(leaky-relu)",lw = 3)   
plt.plot(xs,yss222,label = "sample 0 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2111,label = "sample 0 in training set in QDCNN",lw = 3)
plt.plot(xs,ys2222,label = "sample 0 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys2111e,label = "sample 0 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys2222e,label = "sample 0 in testing set in CNN(rot-relu)",lw = 3)
plt.title("F score-epoch plot of CIFAR100 of sample 0")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xs,ys23,label = "sample 1 in training set in QRCNN",lw = 3)
plt.plot(xs,ys24,label = "sample 1 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys233,label = "sample 1 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys244,label = "sample 1 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2333,label = "sample 1 in training set in QDCNN",lw = 3)
plt.plot(xs,ys2444,label = "sample 1 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys2333e,label = "sample 1 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys2444e,label = "sample 1 in testing set in CNN(rot-relu)",lw = 3)
plt.title("F score-epoch plot of CIFAR100 of sample 1")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(223)
plt.plot(xs,ys25,label = "sample 2 in training set in QRCNN",lw = 3)
plt.plot(xs,ys26,label = "sample 2 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys255,label = "sample 2 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys266,label = "sample 2 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2555,label = "sample 2 in training set in QDCNN",lw = 3)
plt.plot(xs,ys2666,label = "sample 2 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys2555e,label = "sample 2 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys2666e,label = "sample 2 in testing set in CNN(rot-relu)",lw = 3)
plt.title("F score-epoch plot of CIFAR100 of sample 2")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right',prop={'size':9})
plt.grid()

plt.subplot(224)
plt.plot(xs,ys27,label = "sample 3 in training set in QRCNN",lw = 3)
plt.plot(xs,ys28,label = "sample 3 in testing set in QRCNN",lw = 3)
plt.plot(xs,ys277,label = "sample 3 in training set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys288,label = "sample 3 in testing set in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2777,label = "sample 3 in training set in QDCNN",lw = 3)
plt.plot(xs,ys2888,label = "sample 3 in testing set in QDCNN",lw = 3)
plt.plot(xs,ys2777e,label = "sample 3 in training set in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys2888e,label = "sample 3 in testing set in CNN(rot-relu)",lw = 3)
plt.title("F score-epoch plot of CIFAR100 of sample 3")
plt.xlabel("epoch")
plt.ylabel("F score")
plt.legend(loc='lower right')
plt.grid()

In [26]:
%matplotlib qt5
plt.subplot(221)
plt.plot(xs,ys29,label = "weight 1 in QRCNN",lw = 3)
plt.plot(xs,ys30,label = "weight 2 in QRCNN",lw = 3)
plt.plot(xs,ys31,label = "weight 3 in QRCNN",lw = 3)
plt.plot(xs,ys299,label = "weight 1 in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys300,label = "weight 2 in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys311,label = "weight 3 in CNN(leaky-relu)",lw = 3)
plt.plot(xs,ys2999,label = "weight 1 in QDCNN",lw = 3)
plt.plot(xs,ys3000,label = "weight 2 in QDCNN",lw = 3)
plt.plot(xs,ys3111,label = "weight 3 in QDCNN",lw = 3)
plt.plot(xs,ys2999e,label = "weight 1 in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys3000e,label = "weight 2 in CNN(rot-relu)",lw = 3)
plt.plot(xs,ys3111e,label = "weight 3 in CNN(rot-relu)",lw = 3)
plt.title("weight parameter-epoch plot in 4 networks in training set")
plt.xlabel("epoch")
plt.style.use('classic')
plt.ylabel("weight parameter")
plt.legend(loc='lower left',prop={'size':9})
plt.grid()

plt.subplot(222)
plt.plot(xss,ys32,label = "weight 1 grad in QRCNN",lw = 3)
plt.plot(xss,ys33,label = "weight 2 grad in QRCNN",lw = 3)
plt.plot(xss,ys34,label = "weight 3 grad in QRCNN",lw = 3)
plt.plot(xss,ys322,label = "weight 1 grad in CNN(leaky-relu)",lw = 3)
plt.plot(xss,ys333,label = "weight 2 grad in CNN(leaky-relu)",lw = 3)
plt.plot(xss,ys344,label = "weight 3 grad in CNN(leaky-relu)",lw = 3)
plt.plot(xss,ys3222,label = "weight 1 grad in QDCNN",lw = 3)
plt.plot(xss,ys3333,label = "weight 2 grad in QDCNN",lw = 3)
plt.plot(xss,ys3444,label = "weight 3 grad in QDCNN",lw = 3)
plt.plot(xss,ys3222e,label = "weight 1 grad in CNN(rot-relu)",lw = 3)
plt.plot(xss,ys3333e,label = "weight 2 grad in CNN(rot-relu)",lw = 3)
plt.plot(xss,ys3444e,label = "weight 3 grad in CNN(rot-relu)",lw = 3)
plt.title("weight parameter gradient-epoch plot in 4 networks in training set")
plt.style.use('classic')
plt.xlabel("epoch")
plt.ylabel("weight parameter gradient")
plt.legend(loc='lower left',prop={'size':9})
plt.grid()

In [27]:
import xlsxwriter as xw
 
def xw_toExcel(data, fileName):  # xlsxwriter库储存数据到excel
    workbook = xw.Workbook(fileName)  # 创建工作簿
    worksheet1 = workbook.add_worksheet("sheet1")  # 创建子表
    worksheet1.activate()  # 激活表
    title = ['QRCNN', 'CNN(leaky-relu)', 'QDCNN','CNN(rot-relu)']  # 设置表头
    worksheet1.write_row('E1', title)  # 从A1单元格开始写入表头
    i = 2  # 从第二行开始写入数据
    for j in range(len(data)):
        insertData = [data[j]["QRCNN"], data[j]["CNN(leaky-relu)"], data[j]["QDCNN"],data[j]["CNN(rot-relu)"]]
        row = 'E' + str(i)
        worksheet1.write_row(row, insertData)
        i += 1
    workbook.close()  # 关闭表
    
testData = [
{"QRCNN": str(a1[26]), "CNN(leaky-relu)": str(a11[26]), "QDCNN": str(a111[26]), "CNN(rot-relu)":str(a111e[26])},
{"QRCNN": str(a1[24]), "CNN(leaky-relu)": str(a11[24]), "QDCNN": str(a111[24]), "CNN(rot-relu)":str(a111e[24])},
{"QRCNN": str(sum1[26]), "CNN(leaky-relu)": str(sum11[26]), "QDCNN": str(sum111[26]), "CNN(rot-relu)":str(sum111e[26])},
{"QRCNN": str(sum1[24]), "CNN(leaky-relu)": str(sum11[24]), "QDCNN": str(sum111[24]), "CNN(rot-relu)":str(sum111e[24])},
{"QRCNN": str(a1[27]), "CNN(leaky-relu)": str(a11[27]), "QDCNN": str(a111[27]), "CNN(rot-relu)":str(a111e[27])},
{"QRCNN": str(a1[25]), "CNN(leaky-relu)": str(a11[25]), "QDCNN": str(a111[25]), "CNN(rot-relu)":str(a111e[25])},
{"QRCNN": str(sum1[27]), "CNN(leaky-relu)": str(sum11[27]), "QDCNN": str(sum111[27]), "CNN(rot-relu)":str(sum111e[27])},
{"QRCNN": str(sum1[25]), "CNN(leaky-relu)": str(sum11[25]), "QDCNN": str(sum111[25]), "CNN(rot-relu)":str(sum111e[25])},
]
fileName = 'CIFAR result 1 with NO Noise params 1.xlsx'

xw_toExcel(testData, fileName)